# 🫀 PhysioDKS-IncepSE — 12-Lead ECG Multi-Label Classification, Cross-Dataset Generalization & XAI

**Train:** PTB-XL folds 1–8 · **Validate:** PTB-XL fold 9 · **In-domain test:** PTB-XL fold 10 · **Zero-shot external test:** Chapman-Shaoxing (PhysioNet `ecg-arrhythmia` v1.0.0)

> **Is notebook ka maqsad:** ek *publication-grade*, end-to-end, reproducible pipeline jo 4 models ko **bilkul same rules** pe compare kare, aur dikhaye ki proposed **PhysioDKS-IncepSE** doosre hospital ke data pe (bina fine-tuning ke) kitna achha generalize karta hai. Saath hi XAI se dikhaye ki model ECG ke *kis hisse* ko dekh kar decision leta hai.

---

## 0.1 Research problem: asli dikkat kya hai?

| Problem | Matlab (Hinglish) | Is notebook mein solution |
|---|---|---|
| **Domain shift** | Har hospital ki ECG machine, filters, electrode placement aur population alag hoti hai. PTB-XL (Germany, Schiller devices) pe seekha hua model Chapman (China, GE MUSE) pe gir jaata hai. | Identical preprocessing + **Stage-1 PhysioDKS filter + InstanceNorm** |
| **Multi-label imbalance** | Ek ECG mein ek saath kai conditions ho sakti hain (jaise AFIB + RBBB), aur kuch classes bahut rare hain (AFL PTB-XL mein sirf ~0.3%). | Sigmoid multi-label head, per-class threshold tuning, optional Focal/ASL loss |
| **Black-box trust** | Doctor ko pata hona chahiye ki model P/QRS/T waves dekh raha hai ya noise. | 1D Grad-CAM (Stage-1 + final block), SE lead-attention, DKS branch-attention, *quantitative* XAI metrics |

## 0.2 Hypotheses (jo hum test karenge)

- **H1:** Standard CNNs hospital-specific hardware noise aur baseline wander pe overfit karte hain.
- **H2:** *Physiological Multi-Scale DKS Filter* (100 Hz pe QRS/P/T durations ke hisaab se kernels) aur *InstanceNorm* milkar cross-dataset amplitude/offset shift ko suppress karte hain.
- **H3:** Isse **in-domain AUC** aur **zero-shot cross-dataset AUC** dono improve hote hain, ya kam se kam **ΔAUC (generalization gap)** chhota hota hai.
- **H4:** Grad-CAM dikhayega ki proposed model clinically valid waveform regions (P, QRS, T) pe zyada focus karta hai.

> ⚠️ **Research honesty:** Hypothesis tab tak sirf hypothesis hai jab tak numbers support na karein. Ye notebook har claim ke liye **number + 95% CI + paired significance test** deta hai. Agar result hypothesis ke against aaye, to wahi report karna hai. Negative result bhi publishable insight hota hai.

## 0.3 Pipeline overview

```
 PTB-XL (21,837 ECGs, records100 = 100 Hz)          Chapman-Shaoxing (45,152 ECGs, 500 Hz)
            │                                                 │  scipy.signal.resample → 100 Hz
            └──────────────────────┬──────────────────────────┘
                                   ▼
        Bandpass 0.5–40 Hz (Butterworth order-4, zero-phase)  →  per-record per-lead z-score
                                   ▼
          Label harmonization: SCP-ECG / SNOMED-CT  →  unified multi-hot vector  [C classes]
                                   ▼
   PTB-XL folds 1–8 = TRAIN │ fold 9 = VAL (early-stop + thresholds) │ fold 10 = IN-DOMAIN TEST
                                   ▼
     Same engine for all 4 models:  ResNet1D │ InceptionTime │ IncepSE │ PhysioDKS-IncepSE
                                   ▼
   Benchmark: Macro-AUC, F1, Sens, Spec, AUPRC, ΔAUC (+ bootstrap CI)   │   XAI: Grad-CAM, Lead-SE, DKS
```

## 0.4 Notebook map

| Cell | Kya hota hai |
|---|---|
| **1** | Imports, global config, seeds, device |
| **2** | PTB-XL + Chapman metadata, **label harmonization** (SCP-ECG / SNOMED-CT → unified multi-hot), leakage checks |
| **3** | Signal preprocessing (resample → bandpass → z-score), caching, PyTorch `Dataset`/`DataLoader`, sanity checks |
| **4** | 4 model architectures + **dummy-tensor sanity checks** |
| **5** | Training engine (AdamW + OneCycleLR + AMP + grad-clip + early stopping) |
| **6** | Threshold tuning (fold 9) → benchmark table (fold 10 + Chapman), bootstrap CIs, significance tests |
| **7** | XAI: 1D Grad-CAM, SE lead-attention, DKS branch attention, filter spectra, quantitative saliency metrics |

## 0.5 Kaise chalayein (How to run)

1. Ye notebook `coding/` folder mein hai, jahan tumhare baaki notebooks hain. Data paths wahi hain: `../PTBXL dataset/...` aur `../ecg-arrhythmia-1.0.0-chapman/...`.
2. **Pehli baar:** Cell 1 mein `SMOKE_TEST = True` set karke poora notebook chalao (CPU pe bhi ~10 min). Saare cells bina error chal jaayein, tab `False` karke full run karo.
3. **Full run time (GPU):** preprocessing ~5–10 min (sirf ek baar, phir cache se load), training ~15–40 min per model.
4. **Outputs** `./physiodks_runs/<label_set>/full/` mein save hote hain (checkpoints, predictions, tables, figures). Dobara chalane par trained models skip ho jaate hain (`FORCE_RETRAIN = False`).
5. **Dependencies:** `numpy pandas scipy scikit-learn torch wfdb matplotlib tqdm` (tumhare `.venv` mein pehle se hain).

### 📖 Glossary: Section 0 ke naye terms

| Term | Simple Hinglish explanation |
|---|---|
| **ECG / 12-lead ECG** | Dil ki electrical activity ki recording. 12 leads = 12 alag "views": 6 limb leads (I, II, III, aVR, aVL, aVF) + 6 chest leads (V1–V6). |
| **Multi-label classification** | Ek input ke multiple sahi labels ho sakte hain (AFIB **aur** RBBB dono). Multi-*class* mein sirf ek label hota hai. |
| **Domain shift** | Training data aur test data ki distribution alag hona (alag hospital, machine ya population). |
| **Source / target domain** | Source = jahan train kiya (PTB-XL). Target = jahan deploy/test kar rahe ho (Chapman). |
| **Zero-shot cross-dataset evaluation** | Model ko doosre dataset pe *bina kisi training, fine-tuning ya threshold change* ke test karna. |
| **Class imbalance** | Kuch classes ke examples bahut kam (rare) aur kuch ke bahut zyada. |
| **Benchmark** | Same rules pe multiple models ka fair comparison. |
| **Ablation study** | Model ke components ek-ek karke hata/jod ke dekhna ki kaunsa component kitna contribute karta hai. |
| **XAI (Explainable AI)** | Techniques jo batati hain ki model ne decision *kyun* liya. |
| **Hypothesis** | Ek testable claim jo experiment se support ya reject hota hai. |

---
# Cell 1 — Imports & Global Config ⚙️

**Is cell mein kya hota hai?**
1. Saari libraries import hoti hain (NumPy, SciPy, PyTorch, wfdb, scikit-learn, matplotlib).
2. Ek `Config` dataclass mein **saare hyper-parameters ek jagah** rakhe hain, taaki har model *exactly same* settings pe chale (benchmark parity).
3. Random seed fix hota hai (reproducibility) aur device (GPU/CPU) choose hota hai.

**Sabse important switches:**

| Switch | Default | Kab change karein |
|---|---|---|
| `SMOKE_TEST` | `False` | Pehli baar `True` karo → chhote subset pe 2 epochs, sirf pipeline test |
| `LABEL_SET` | `'rhythm9'` | `'superclass5'` = PTB-XL ke 5 diagnostic superclasses (in-domain ablation) |
| `RUN_EXTENDED_ABLATION` | `False` | `True` → Stage-1 ke har component ka alag contribution (4 extra models) |
| `FORCE_RETRAIN` | `False` | `True` → purane checkpoints ignore karke dobara train |

> 💡 **Engineering insight:** Config ko code se alag rakhna (*single source of truth*) research mein bahut zaroori hai. Paper ke "Implementation details" section ke saare numbers yahin se aate hain, aur galti se kisi ek model ko alag learning-rate dene jaisi unfair comparison nahi ho paati.

> 🪟 **Windows note:** `num_workers = 0` rakha hai. Data RAM / memory-map mein hai, isliye extra worker processes ki zaroorat nahi, aur Jupyter + Windows ka multiprocessing crash bhi avoid ho jaata hai.

### 📖 Glossary: Cell 1

| Term | Hinglish explanation |
|---|---|
| **Hyper-parameter** | Woh settings jo hum training se *pehle* choose karte hain (learning rate, batch size, epochs). Model inhe khud nahi seekhta. |
| **Random seed** | Random number generator ka starting point. Same seed ⇒ same "random" numbers ⇒ same result (reproducibility). |
| **Deterministic mode** | GPU ke kuch fast algorithms har run mein thoda alag result dete hain. Deterministic mode unhe band karke result repeatable banata hai (thoda slow). |
| **CUDA / GPU** | NVIDIA GPU pe parallel computation. Deep learning training 10–50× fast ho jaati hai. |
| **AMP (Automatic Mixed Precision)** | Zyada-tar operations float16 (half precision) mein → kam memory aur fast training. Sensitive operations (loss, normalization) float32 mein hi rehte hain. |
| **Dataclass** | Python ka feature jo config jaisi "data holder" classes ko clean banata hai. |
| **Cache** | Ek baar ka heavy computation (preprocessing) disk pe save karna taaki agli baar seedha load ho. |
| **Smoke test** | "Kahin dhuaan to nahi nikal raha?" Chhote data pe quick run, sirf yeh check karne ke liye ki code crash nahi hota. |

In [ ]:
# =====================================================================
# Cell 1: Imports & Global Config
# =====================================================================
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS: torch import se PEHLE set hona chahiye

import ast, json, math, random, sys, time, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy
import scipy.io
from scipy.signal import butter, find_peaks, resample, sosfiltfilt, welch
from scipy.stats import wilcoxon
try:                                    # SciPy >= 1.15 mein naya naam
    from scipy.signal import freqz_sos as sos_freqz
except ImportError:                     # purane SciPy mein
    from scipy.signal import sosfreqz as sos_freqz
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
import wfdb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.colors import Normalize

try:
    from tqdm.auto import tqdm
except ImportError:                     # tqdm na ho to simple fallback
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ---------------------------------------------------------------------
# 1.1 Run-mode switches (yahi sabse zyada change karoge)
# ---------------------------------------------------------------------
SMOKE_TEST = False             # True = chhota subset + 2 epochs. Numbers MEANINGLESS, sirf pipeline check.
LABEL_SET = 'rhythm9'          # 'rhythm9' (main cross-dataset task) | 'superclass5' (PTB-XL superclass ablation)
RUN_EXTENDED_ABLATION = False  # True = Stage-1 component ablation (noDKS / noIN / noLeadSE / randInit)
FORCE_RETRAIN = False          # True = existing checkpoints ignore karke dobara train

# Optional environment-variable overrides (CI/testing ke liye; normal use mein ignore karo)
SMOKE_TEST = os.environ.get('PHYSIODKS_SMOKE', str(int(SMOKE_TEST))) == '1'
LABEL_SET = os.environ.get('PHYSIODKS_LABEL_SET', LABEL_SET)
RUN_EXTENDED_ABLATION = os.environ.get('PHYSIODKS_EXT_ABL', str(int(RUN_EXTENDED_ABLATION))) == '1'


# ---------------------------------------------------------------------
# 1.2 Config: single source of truth (saare models ke liye LOCKED)
# ---------------------------------------------------------------------
@dataclass
class Config:
    # ---------- Paths (tumhara existing folder structure, kuch change nahi) ----------
    ptbxl_path: Path = Path('../PTBXL dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/')
    chapman_path: Path = Path('../ecg-arrhythmia-1.0.0-chapman/'
                              'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/')
    cache_dir: Path = Path('./physiodks_cache')     # preprocessed signals (.npy)
    run_dir: Path = Path('./physiodks_runs')        # checkpoints / predictions / tables / figures
    # ---------- LOCKED signal protocol ----------
    fs: int = 100                                    # target sampling rate (Hz)
    duration_s: int = 10                             # 10 s  =>  1000 samples
    n_leads: int = 12
    bandpass_hz: Tuple[float, float] = (0.5, 40.0)
    filter_order: int = 4
    flat_lead_std_mv: float = 1e-4                   # isse kam std wali lead = flat/missing  =>  zeros
    ptbxl_from_500hz: bool = False                   # True => PTB-XL bhi 500 Hz files se same resampler se (full parity, slow)
    # ---------- Labels ----------
    label_set: str = 'rhythm9'
    min_pos: Dict[str, int] = field(default_factory=lambda: {'train': 50, 'val': 5, 'test': 5, 'ext': 30})
    # ---------- LOCKED training protocol ----------
    seeds: Tuple[int, ...] = (42,)                   # multi-seed ke liye: (42, 43, 44)
    epochs: int = 40
    patience: int = 8                                # early-stopping patience (epochs)
    batch_size: int = 64
    eval_batch_size: int = 256
    max_lr: float = 1e-3
    pct_start: float = 0.2
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    loss_name: str = 'bce'                           # 'bce' | 'focal' | 'asl'
    use_amp: bool = True
    deterministic: bool = True
    num_workers: int = 0                             # data RAM/memmap mein hai; 0 = Windows/Jupyter-safe
    force_retrain: bool = False
    # ---------- Models ----------
    models: Tuple[str, ...] = ('ResNet1D', 'InceptionTime', 'IncepSE', 'PhysioDKS-IncepSE')
    dropout: float = 0.3
    se_squeeze: str = 'avgstd'                       # SE squeeze: 'avg' (original paper) | 'avgstd' (mean+std, robust)
    dks_filters_per_lead: int = 4                    # m: har lead ke liye har branch mein kitne filters
    # ---------- Evaluation ----------
    n_boot: int = 1000
    boot_seed: int = 2024
    # ---------- XAI ----------
    xai_n_per_class: int = 100                       # quantitative XAI: har class ke kitne Chapman positives
    xai_n_examples: int = 2                          # har class ke kitne side-by-side example plots
    # ---------- Smoke test ----------
    smoke_test: bool = False
    smoke_n: Dict[str, int] = field(default_factory=lambda: {'train': 600, 'val': 200, 'test': 200, 'ext': 800})

    @property
    def n_samples(self) -> int:
        return self.fs * self.duration_s             # 100 Hz * 10 s = 1000

    @property
    def mode_tag(self) -> str:
        return 'smoke' if self.smoke_test else 'full'


CFG = Config(label_set=LABEL_SET, smoke_test=SMOKE_TEST, force_retrain=FORCE_RETRAIN)
if RUN_EXTENDED_ABLATION:
    CFG.models = CFG.models + ('PhysioDKS-noDKS', 'PhysioDKS-noIN', 'PhysioDKS-noLeadSE', 'PhysioDKS-randInit')
if CFG.smoke_test:
    CFG.epochs, CFG.patience, CFG.n_boot = 2, 2, 50
    CFG.min_pos = {'train': 1, 'val': 1, 'test': 1, 'ext': 1}
    CFG.xai_n_per_class, CFG.xai_n_examples = 16, 1
CFG.epochs = int(os.environ.get('PHYSIODKS_EPOCHS', CFG.epochs))

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
RUN_DIR = CFG.run_dir / CFG.label_set / CFG.mode_tag
FIG_DIR, TAB_DIR = RUN_DIR / 'figures', RUN_DIR / 'tables'
for _d in (CFG.cache_dir, RUN_DIR, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# 1.3 Device, determinism, seeds
# ---------------------------------------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CFG.use_amp and DEVICE.type == 'cuda'      # AMP sirf CUDA pe


def seed_everything(seed: int, deterministic: bool = True) -> None:
    """Python, NumPy aur PyTorch (CPU + GPU) sab ke random generators ko same seed do."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)   # non-deterministic op pe sirf warning


seed_everything(CFG.seeds[0], CFG.deterministic)


# ---------------------------------------------------------------------
# 1.4 Plot style (validated colour-blind-safe palette; colour = entity, har chart mein same)
# ---------------------------------------------------------------------
INK, INK2, GRID, AXIS = '#191919', '#666666', '#e7e7e7', '#c2c2c2'
MODEL_COLORS = {
    'ResNet1D': '#f2a600', 'InceptionTime': '#00b894', 'IncepSE': '#f47230', 'PhysioDKS-IncepSE': '#257de0',
    'PhysioDKS-noDKS': '#ee7fac', 'PhysioDKS-noIN': '#008a17', 'PhysioDKS-noLeadSE': '#6a4ce2',
    'PhysioDKS-randInit': '#ec4647',
}
DATASET_COLORS = {'PTB-XL': '#6a4ce2', 'Chapman': '#ec4647'}
BRANCH_COLORS = ['#6a4ce2', '#ee7fac', '#008a17']    # Stage-1 branches k=5 / 11 / 21
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'font.family': 'sans-serif',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'axes.edgecolor': AXIS,
    'axes.labelcolor': INK2, 'axes.titlecolor': INK, 'text.color': INK, 'xtick.color': INK2,
    'ytick.color': INK2, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.6, 'legend.frameon': False,
})


def savefig(fig: plt.Figure, name: str) -> None:
    """Figure ko 300-dpi PNG mein save karo aur notebook mein dikhao."""
    fig.savefig(FIG_DIR / f'{name}.png')
    plt.show()
    plt.close(fig)


def show(df: pd.DataFrame, precision: int = 4) -> None:
    with pd.option_context('display.max_columns', 60, 'display.width', 250, 'display.precision', precision):
        display(df)


# ---------------------------------------------------------------------
# 1.5 Environment summary
# ---------------------------------------------------------------------
print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | numpy {np.__version__} | '
      f'scipy {scipy.__version__} | wfdb {wfdb.__version__} | pandas {pd.__version__}')
gpu = f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''
print(f'Device: {DEVICE}{gpu} | AMP: {USE_AMP} | deterministic: {CFG.deterministic}')
print(f'Label set: {CFG.label_set} | models: {list(CFG.models)} | seeds: {CFG.seeds} | epochs: {CFG.epochs}')
print(f'Outputs -> {RUN_DIR.resolve()}')
for _name, _p in [('PTB-XL', CFG.ptbxl_path), ('Chapman', CFG.chapman_path)]:
    print(f'{_name:8s}: {_p.resolve()}  {"✅ found" if _p.exists() else "❌ NOT FOUND (Config path check karo)"}')
if CFG.smoke_test:
    print('\n' + '#' * 78 + '\n#  SMOKE TEST MODE: chhota subset, 2 epochs. Saare numbers MEANINGLESS hain.  #\n' + '#' * 78)

---
# Cell 2 — Data Loading & Label Harmonization 🏷️

## 2.1 Do datasets, do alag label "languages"

| | **PTB-XL** (source domain) | **Chapman-Shaoxing** (target domain) |
|---|---|---|
| Source | PTB, Germany (Schiller AG devices), 1989–1996 | Chapman Univ. + Shaoxing People's Hospital (+ Ningbo First Hospital), China (GE MUSE) |
| Records | 21,837 ECGs (18,885 patients) | 45,152 ECGs (PhysioNet `ecg-arrhythmia` v1.0.0) |
| Sampling | 100 Hz (`records100/`) + 500 Hz (`records500/`) | 500 Hz (`.mat`, int16, gain 1000/mV) |
| Label "language" | **SCP-ECG** statements (`'AFIB'`, `'CRBBB'`, ...) | **SNOMED-CT** codes (`164889003`, `59118001`, ...) in header line `#Dx:` |
| Is study mein use | folds 1–8 train · fold 9 val · fold 10 in-domain test | **sirf zero-shot external test** |

> ℹ️ PhysioNet ka ye 45,152-record release asal mein CinC-2021 ke *Chapman-Shaoxing* (10,247) + *Ningbo* (34,905) sources ka combination hai. Yaani "external" set khud do hospitals ka mix hai. Paper mein ye mention karna achha rahega.

## 2.2 Label harmonization: kyun aur kaise?

Model ek **fixed output vector** predict karta hai (jaise 9 numbers). PTB-XL mein "AFIB" likha hai aur Chapman mein "164889003", to dono ko **ek hi unified class** pe map karna padega. Iske bina cross-dataset evaluation meaningless hai.

**Label set A: `rhythm9` (main experiment).** PhysioNet/CinC-2021 Challenge ke woh SNOMED-CT scored classes jo *dono* datasets mein exist karte hain:

| Unified | Clinical meaning (1 line) | SNOMED-CT | PTB-XL SCP | Chapman code (acronym) |
|---|---|---|---|---|
| **NSR** | Normal sinus rhythm: 60–100 bpm, har QRS se pehle P-wave | 426783006 | `SR` | 426783006 (SR) |
| **AFIB** | Atrial fibrillation: RR irregularly irregular, P-waves gayab, fibrillatory baseline | 164889003 | `AFIB` | 164889003 (AFIB) |
| **AFL** | Atrial flutter: "sawtooth" F-waves (~300/min) | 164890007 | `AFLT` | 164890007 (**`AF`**!) |
| **RBBB** | Right bundle branch block: wide QRS, V1 mein rSR′ | 59118001 ≡ 713427006 | `CRBBB` | 59118001 (RBBB) |
| **LBBB** | Left bundle branch block: wide QRS, I/V6 mein broad notched R | 164909002 ≡ 733534002 | `CLBBB` | 164909002 (LBBB) |
| **PAC** | Premature atrial contraction: jaldi aaya beat, alag shape ki P | 284470004 ≡ 63593006 | `PAC` | 284470004 (APB) |
| **PVC** | Premature ventricular contraction: jaldi aaya *wide* QRS, P nahi | 427172004 ≡ 17338001 | `PVC` | 17338001 (VPB) |
| **STach** | Sinus tachycardia: > 100 bpm | 427084000 | `STACH` | 427084000 (ST) |
| **SBrady** | Sinus bradycardia: < 60 bpm | 426177001 | `SBRAD` | 426177001 (SB) |

(≡ ka matlab: CinC-2021 ne in codes ko equivalent score kiya tha, isliye merge kiye.)

**Label set B: `superclass5` (in-domain ablation).** PTB-XL ke 5 diagnostic superclasses (NORM, MI, STTC, CD, HYP). PTB-XL side official `scp_statements.csv` recipe se banta hai. Chapman side ek **approximate** SNOMED → superclass mapping hai (Tier-B: definitions perfectly match nahi karti). Isliye cross-dataset claims ke liye `rhythm9` primary hai.

## 2.3 ⚠️ Known label caveats (paper ke "Limitations" section ke layak)

1. **Chapman acronym trap:** `AF` = atrial **flutter**, `AFIB` = fibrillation. Hum SNOMED codes se map karte hain, acronyms se nahi, isliye ye confusion yahan nahi hota.
2. **Shared SNOMED code:** Chapman ki `ConditionNames_SNOMED-CT.csv` mein `LBBB`, `LBBBB` (posterior fascicular) aur `LFBBB` (anterior fascicular), teeno ka code `164909002` hai. Isliye Chapman ka LBBB label thoda noisy ho sakta hai.
3. **Strict ectopy mapping:** `VEB` (ventricular *escape* beat) PVC nahi hai (escape beat late aata hai, PVC early), isliye use include nahi kiya.
4. **PTB-XL likelihoods:** rhythm statements (SR, AFIB, ...) ki likelihood hamesha `0.0` hoti hai. Isliye hum likelihood filter **nahi** lagate, warna saare rhythm labels gayab ho jaate.
5. **Class selection rule:** ek class tabhi rakhi jaati hai jab train / val / test / Chapman sab mein minimum positives hon (`CFG.min_pos`). Ye rule sirf *label availability* pe based hai, model performance pe nahi, isliye isme leakage nahi hai.

## 2.4 Patient-stratified split (data leakage se bachav)

PTB-XL ka `strat_fold` column authors ne **patient-wise** banaya hai: ek patient ke saare ECG ek hi fold mein jaate hain. Hum code mein `assert` lagakar verify bhi karte hain ki train/val/test mein koi patient common nahi hai. Chapman sirf external test hai: ise kabhi training, early stopping, normalization statistics ya threshold tuning mein touch nahi kiya jaata.

### 📖 Glossary: Cell 2

| Term | Hinglish explanation |
|---|---|
| **WFDB format** | PhysioNet ka standard format: `.hea` (header = metadata text) + `.dat`/`.mat` (signal ke numbers). |
| **SCP-ECG** | ECG statements ka European standard (PTB-XL ke 71 codes, jaise `NORM`, `IMI`, `AFIB`). |
| **SNOMED-CT** | Duniya ka sabse bada clinical terminology system: har diagnosis ka unique numeric code. |
| **Label harmonization** | Alag datasets ke label naamon ko ek common label space pe map karna. |
| **Multi-hot encoding** | Har record ke liye 0/1 vector; jitni conditions, utne 1 (jaise `[1,0,0,1,0,...]`). One-hot mein sirf ek 1 hota hai. |
| **Superclass** | Bahut saare detailed codes ka bada group (jaise IMI, ASMI, ... → MI). |
| **`strat_fold`** | PTB-XL ka 1–10 fold number. Stratified = har fold mein labels ka distribution similar. |
| **Patient-stratified split** | Split patient ke level pe hota hai: ek patient train aur test dono mein nahi ho sakta. |
| **Data leakage** | Test ki information kisi bhi tarah training/tuning mein ghus jaana. Results jhoothe-achhe dikhne lagte hain. |
| **Prevalence** | Kisi class ke positive records ka percentage. |
| **Label (prior) shift** | Do datasets mein class prevalence alag hona. AUC pe iska asar kam hota hai, F1/AUPRC pe zyada. |
| **Tier-A / Tier-B mapping** | Tier-A = definitions closely match karti hain; Tier-B = naam same hai par diagnostic criteria alag ho sakte hain. |

In [ ]:
# =====================================================================
# Cell 2.1: Label harmonization dictionaries (clean + documented)
# =====================================================================
# ---------- Label set A: 'rhythm9' (CinC-2021 SNOMED-CT scored classes jo dono datasets mein hain) ----------
# Har unified class ke liye: SNOMED-CT reference | PTB-XL SCP-ECG code(s) | Chapman SNOMED code(s) (header '#Dx:')
RHYTHM9_MAP: Dict[str, Dict[str, set]] = {
    'NSR':    {'snomed': {'426783006'},              'ptbxl': {'SR'},    'chapman': {'426783006'}},              # sinus rhythm
    'AFIB':   {'snomed': {'164889003'},              'ptbxl': {'AFIB'},  'chapman': {'164889003'}},              # atrial fibrillation
    'AFL':    {'snomed': {'164890007'},              'ptbxl': {'AFLT'},  'chapman': {'164890007'}},              # atrial flutter (Chapman acronym 'AF'!)
    'RBBB':   {'snomed': {'59118001', '713427006'},  'ptbxl': {'CRBBB'}, 'chapman': {'59118001', '713427006'}},  # RBBB ≡ complete RBBB
    'LBBB':   {'snomed': {'164909002', '733534002'}, 'ptbxl': {'CLBBB'}, 'chapman': {'164909002', '733534002'}}, # LBBB ≡ complete LBBB
    'PAC':    {'snomed': {'284470004', '63593006'},  'ptbxl': {'PAC'},   'chapman': {'284470004', '63593006'}},  # PAC ≡ SVPB (Chapman 'APB')
    'PVC':    {'snomed': {'427172004', '17338001'},  'ptbxl': {'PVC'},   'chapman': {'427172004', '17338001'}},  # PVC ≡ VPB (Chapman 'VPB')
    'STach':  {'snomed': {'427084000'},              'ptbxl': {'STACH'}, 'chapman': {'427084000'}},              # sinus tachycardia
    'SBrady': {'snomed': {'426177001'},              'ptbxl': {'SBRAD'}, 'chapman': {'426177001'}},              # sinus bradycardia
}
RHYTHM9_CLASSES: List[str] = list(RHYTHM9_MAP)
PTBXL_SCP_TO_RHYTHM9: Dict[str, str] = {scp: c for c, d in RHYTHM9_MAP.items() for scp in d['ptbxl']}
CHAPMAN_SNOMED_TO_RHYTHM9: Dict[str, str] = {code: c for c, d in RHYTHM9_MAP.items() for code in d['chapman']}

# ---------- Label set B: 'superclass5' (PTB-XL diagnostic superclasses) ----------
SUPERCLASS5_CLASSES: List[str] = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
# PTB-XL side: scp_statements.csv ke 'diagnostic_class' column se (official PTB-XL recipe, Cell 2.3 mein).
# Chapman side: APPROXIMATE (Tier-B) mapping. Comments mein Chapman CSV ke apne acronyms/naam.
CHAPMAN_SNOMED_TO_SUPERCLASS5: Dict[str, str] = {
    # CD: Conduction Disturbance
    '270492004': 'CD',   # 1AVB   1st-degree AV block
    '195042002': 'CD',   # 2AVB   2nd-degree AV block
    '54016002':  'CD',   # 2AVB1  2nd-degree AV block (Mobitz I)
    '28189009':  'CD',   # 2AVB2  2nd-degree AV block (Mobitz II)
    '27885002':  'CD',   # 3AVB   complete AV block
    '233917008': 'CD',   # AVB    AV block (unspecified)
    '164909002': 'CD',   # LBBB / LBBBB / LFBBB (CSV mein teeno ka same code)
    '59118001':  'CD',   # RBBB
    '698252002': 'CD',   # IVB / IDC (intraventricular conduction delay)
    '74390002':  'CD',   # WPW
    # HYP: Hypertrophy
    '164873001': 'HYP',  # LVH
    '89792004':  'HYP',  # RVH
    '446358003': 'HYP',  # RAH (right atrial hypertrophy)
    # MI: Myocardial Infarction
    '164865005': 'MI',   # MI (MIBW / MIFW / MILW / MISW: sab walls ka same code)
    # STTC: ST/T change
    '428750005': 'STTC', # STTC   ST-T change
    '429622005': 'STTC', # STDD   'ST drop down' (ST depression)
    '164930006': 'STTC', # STE    'ST extension' (CSV naming)
    '164931005': 'STTC', # STTU   'ST tilt up' (CSV naming)
    '164934002': 'STTC', # TWC    T-wave change
    '59931005':  'STTC', # TWO    'T wave opposite' (inversion)
    '111975006': 'STTC', # QTIE   QT interval extension (PTB-XL mein LNGQT bhi STTC hai)
}
CHAPMAN_NORM_PROXY = '426783006'   # Chapman mein 'normal ECG' code nahi hai: NORM ≈ SR + koi CD/HYP/MI/STTC code nahi

print('rhythm9     :', RHYTHM9_CLASSES)
print('superclass5 :', SUPERCLASS5_CLASSES)
print(f'PTB-XL SCP -> rhythm9 lookup     : {PTBXL_SCP_TO_RHYTHM9}')
print(f'Chapman SNOMED -> rhythm9 lookup : {len(CHAPMAN_SNOMED_TO_RHYTHM9)} codes | '
      f'Chapman SNOMED -> superclass5: {len(CHAPMAN_SNOMED_TO_SUPERCLASS5)} codes')

In [ ]:
# =====================================================================
# Cell 2.2: Metadata loading (PTB-XL CSV + Chapman WFDB headers)
# =====================================================================
def load_ptbxl_metadata(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """PTB-XL database table + SCP statements table."""
    db_file = cfg.ptbxl_path / 'ptbxl_database.csv'
    if not db_file.exists():
        raise FileNotFoundError(f'PTB-XL nahi mila: {db_file.resolve()} (Config.ptbxl_path check karo)')
    db = pd.read_csv(db_file, index_col='ecg_id')
    db['scp_codes'] = db['scp_codes'].apply(ast.literal_eval)          # "{'NORM': 100.0, 'SR': 0.0}" -> dict
    scp = pd.read_csv(cfg.ptbxl_path / 'scp_statements.csv', index_col=0)
    return db, scp


def parse_wfdb_header(hea_path: Path) -> dict:
    """Chapman .hea file ko plain text ki tarah parse karo (wfdb se fast aur robust).

    Line 1        : '<record> <n_sig> <fs> <n_samples>'
    Next n_sig    : '<file> <fmt> <gain>[(baseline)]/mV <adc_res> <adc_zero> <init> <checksum> <block> <lead>'
    Comment lines : '#Age: 85', '#Sex: Male', '#Dx: 164889003,59118001,164934002', ...
    """
    with open(hea_path, 'r', errors='ignore') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    record, n_sig, fs, n_samp = lines[0].split()[:4]
    n_sig = int(n_sig)
    gains, baselines, leads = [], [], []
    for ln in lines[1:1 + n_sig]:
        parts = ln.split()
        g = parts[2].split('/')[0]                                     # '1000' ya '1000(0)'
        if '(' in g:
            gains.append(float(g.split('(')[0]))
            baselines.append(float(g.split('(')[1].rstrip(')')))
        else:                                                          # baseline na likha ho => ADC zero
            gains.append(float(g))
            baselines.append(float(parts[4]) if len(parts) > 4 else 0.0)
        leads.append(parts[-1])
    meta = {'record': record, 'fs': float(fs), 'n_samples': int(n_samp), 'n_sig': n_sig,
            'gains': ';'.join(map(str, gains)), 'baselines': ';'.join(map(str, baselines)), 'leads': ';'.join(leads)}
    for ln in lines:
        if ln.startswith('#') and ':' in ln:
            key, val = ln[1:].split(':', 1)
            meta[key.strip().lower()] = val.strip()
    return meta


def load_chapman_metadata(cfg: Config) -> pd.DataFrame:
    """Saare Chapman headers parse karke ek table banao (pehli baar ~1 min, phir cached CSV)."""
    cache_file = cfg.cache_dir / 'chapman_headers.csv'
    if cache_file.exists():
        return pd.read_csv(cache_file, dtype={'record': str, 'dx': str}, keep_default_na=False)
    root = cfg.chapman_path / 'WFDBRecords'
    if not root.exists():
        raise FileNotFoundError(f'Chapman WFDBRecords nahi mila: {root.resolve()} (Config.chapman_path check karo)')
    rows = []
    for p in tqdm(sorted(root.rglob('*.hea')), desc='Chapman headers'):
        m = parse_wfdb_header(p)
        m['rel_path'] = p.relative_to(cfg.chapman_path).with_suffix('').as_posix()   # OS-independent path
        rows.append(m)
    df = pd.DataFrame(rows)
    for col in ('dx', 'age', 'sex'):
        if col not in df:
            df[col] = ''
    df = df.fillna('')
    df.to_csv(cache_file, index=False)
    return df


ptb_db, scp_df = load_ptbxl_metadata(CFG)
chap_df = load_chapman_metadata(CFG)
print(f'PTB-XL : {len(ptb_db):,} records | {ptb_db.patient_id.nunique():,} patients | '
      f'folds {sorted(ptb_db.strat_fold.unique().tolist())}')
print(f'Chapman: {len(chap_df):,} header files | fs: {chap_df.fs.value_counts().to_dict()} | '
      f'n_samples: {chap_df.n_samples.value_counts().head(3).to_dict()}')


def smoke_subset(cfg: Config, db: pd.DataFrame, cdf: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """SMOKE_TEST: signals load karne se PEHLE chhota subset chuno (fold structure same rehta hai)."""
    rng = np.random.default_rng(0)
    f = db.strat_fold.values

    def pick(pool: np.ndarray, n: int) -> np.ndarray:
        return rng.choice(pool, size=min(n, len(pool)), replace=False)

    keep = np.concatenate([pick(np.where(f <= 8)[0], cfg.smoke_n['train']),
                           pick(np.where(f == 9)[0], cfg.smoke_n['val']),
                           pick(np.where(f == 10)[0], cfg.smoke_n['test'])])
    ext = np.sort(pick(np.arange(len(cdf)), cfg.smoke_n['ext']))
    return db.iloc[np.sort(keep)], cdf.iloc[ext].reset_index(drop=True)


if CFG.smoke_test:
    ptb_db, chap_df = smoke_subset(CFG, ptb_db, chap_df)
    print(f'[SMOKE] subset -> PTB-XL {len(ptb_db):,} | Chapman {len(chap_df):,}')

In [ ]:
# =====================================================================
# Cell 2.3: Multi-hot labels, class selection, leakage checks, label-shift view
# =====================================================================
def ptbxl_label_matrix(db: pd.DataFrame, scp: pd.DataFrame, label_set: str) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Returns Y [N, C] multi-hot, keep-mask [N], class names."""
    if label_set == 'rhythm9':
        classes = RHYTHM9_CLASSES
        Y = np.zeros((len(db), len(classes)), dtype=np.float32)
        for i, codes in enumerate(db.scp_codes.values):
            for code in codes:                  # likelihood ignore (rhythm codes ki likelihood 0.0 hoti hai)
                cls = PTBXL_SCP_TO_RHYTHM9.get(code)
                if cls is not None:
                    Y[i, classes.index(cls)] = 1.0
        keep = np.ones(len(db), dtype=bool)     # all-negative records bhi valid negatives hain
    elif label_set == 'superclass5':
        classes = SUPERCLASS5_CLASSES
        diag_class = scp.loc[scp.diagnostic == 1, 'diagnostic_class']   # 44 diagnostic SCP codes -> superclass
        Y = np.zeros((len(db), len(classes)), dtype=np.float32)
        for i, codes in enumerate(db.scp_codes.values):
            for code in codes:
                if code in diag_class.index:
                    Y[i, classes.index(diag_class[code])] = 1.0
        keep = Y.sum(1) > 0                     # official recipe: bina superclass wale records drop
    else:
        raise ValueError(f'Unknown label_set: {label_set}')
    return Y, keep, classes


def chapman_label_matrix(cdf: pd.DataFrame, label_set: str) -> Tuple[np.ndarray, np.ndarray, List[set]]:
    """Returns Y [N, C] multi-hot, keep-mask [N], har record ke SNOMED code sets."""
    code_sets = [{c.strip() for c in str(dx).split(',') if c.strip()} for dx in cdf.dx.values]
    if label_set == 'rhythm9':
        classes, lookup = RHYTHM9_CLASSES, CHAPMAN_SNOMED_TO_RHYTHM9
    else:
        classes, lookup = SUPERCLASS5_CLASSES, CHAPMAN_SNOMED_TO_SUPERCLASS5
    Y = np.zeros((len(cdf), len(classes)), dtype=np.float32)
    for i, codes in enumerate(code_sets):
        for code in codes:
            cls = lookup.get(code)
            if cls is not None:
                Y[i, classes.index(cls)] = 1.0
        if label_set == 'superclass5' and Y[i].sum() == 0 and CHAPMAN_NORM_PROXY in codes:
            Y[i, classes.index('NORM')] = 1.0
    keep = np.ones(len(cdf), dtype=bool) if label_set == 'rhythm9' else Y.sum(1) > 0
    return Y, keep, code_sets


Y_ptb_all, keep_ptb, CLASSES_ALL = ptbxl_label_matrix(ptb_db, scp_df, CFG.label_set)
Y_chap_all, keep_chap, chap_code_sets = chapman_label_matrix(chap_df, CFG.label_set)

# ---------- Patient-stratified splits (PTB-XL official strat_fold) ----------
fold = ptb_db.strat_fold.values
idx_train = np.where((fold <= 8) & keep_ptb)[0]
idx_val = np.where((fold == 9) & keep_ptb)[0]
idx_test = np.where((fold == 10) & keep_ptb)[0]

pid = ptb_db.patient_id.values
_pats = {'train': set(pid[idx_train]), 'val': set(pid[idx_val]), 'test': set(pid[idx_test])}
_overlap = {f'{a}∩{b}': len(_pats[a] & _pats[b]) for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]}
assert all(v == 0 for v in _overlap.values()), f'DATA LEAKAGE! Patient overlap mila: {_overlap}'
print(f'✅ Patient-level leakage check passed {_overlap} | '
      f'train {len(idx_train):,} / val {len(idx_val):,} / test {len(idx_test):,} records')

# ---------- Class selection (sirf label availability pe based, performance pe nahi) ----------
counts = pd.DataFrame({
    'PTB-XL train (f1-8)': Y_ptb_all[idx_train].sum(0),
    'PTB-XL val (f9)': Y_ptb_all[idx_val].sum(0),
    'PTB-XL test (f10)': Y_ptb_all[idx_test].sum(0),
    'Chapman (ext)': Y_chap_all[keep_chap].sum(0),
}, index=CLASSES_ALL).astype(int)
counts['PTB-XL prev %'] = 100 * Y_ptb_all[keep_ptb].mean(0)
counts['Chapman prev %'] = 100 * Y_chap_all[keep_chap].mean(0)
mp = CFG.min_pos
counts['selected'] = ((counts['PTB-XL train (f1-8)'] >= mp['train']) & (counts['PTB-XL val (f9)'] >= mp['val'])
                      & (counts['PTB-XL test (f10)'] >= mp['test']) & (counts['Chapman (ext)'] >= mp['ext']))
show(counts.round(2))
counts.to_csv(TAB_DIR / 'label_counts.csv')

CLASSES: List[str] = [c for c in CLASSES_ALL if counts.loc[c, 'selected']]
_dropped = [c for c in CLASSES_ALL if c not in CLASSES]
if _dropped:
    print(f'⚠️ Dropped (min_pos {mp} fail): {_dropped}')
_cidx = [CLASSES_ALL.index(c) for c in CLASSES]
Y_ptb, Y_chap = Y_ptb_all[:, _cidx], Y_chap_all[:, _cidx]
N_CLASSES = len(CLASSES)
assert N_CLASSES >= 2, 'Kam se kam 2 classes chahiye. min_pos ya label_set check karo.'
print(f'Final label space ({N_CLASSES}): {CLASSES}')
print(f'Records with >1 label -> PTB-XL {(Y_ptb[keep_ptb].sum(1) > 1).mean():.1%} | '
      f'Chapman {(Y_chap[keep_chap].sum(1) > 1).mean():.1%}  (isliye MULTI-label: sigmoid + BCE)')
print(f'All-negative records  -> PTB-XL {(Y_ptb[keep_ptb].sum(1) == 0).mean():.1%} | '
      f'Chapman {(Y_chap[keep_chap].sum(1) == 0).mean():.1%}  (valid negatives, kept)')

# ---------- Transparency: Chapman ke woh codes jo kisi class pe map nahi hue ----------
SNOMED_NAMES: Dict[str, str] = {}
_cond_file = CFG.chapman_path / 'ConditionNames_SNOMED-CT.csv'
if _cond_file.exists():
    _cn = pd.read_csv(_cond_file)
    _acr = next(c for c in _cn.columns if 'acronym' in c.lower())
    _sn = next(c for c in _cn.columns if 'snomed' in c.lower())
    for _code, _grp in _cn.groupby(_cn[_sn].astype(str).str.strip()):
        SNOMED_NAMES[_code] = '/'.join(_grp[_acr].astype(str))
_lookup = CHAPMAN_SNOMED_TO_RHYTHM9 if CFG.label_set == 'rhythm9' else CHAPMAN_SNOMED_TO_SUPERCLASS5
_unmapped = pd.Series([c for s in chap_code_sets for c in s if c not in _lookup]).value_counts().head(12)
print('\nChapman codes NOT in this label space (top 12):')
print(pd.DataFrame({'count': _unmapped, 'acronym': [SNOMED_NAMES.get(c, '?') for c in _unmapped.index]}).to_string())

# ---------- Label-shift view: prevalence dumbbell (log scale) ----------
fig, ax = plt.subplots(figsize=(8, 0.45 * N_CLASSES + 1.4))
yy = np.arange(N_CLASSES)
p_ptb = counts.loc[CLASSES, 'PTB-XL prev %'].clip(lower=1e-2).values
p_chp = counts.loc[CLASSES, 'Chapman prev %'].clip(lower=1e-2).values
ax.hlines(yy, np.minimum(p_ptb, p_chp), np.maximum(p_ptb, p_chp), color=AXIS, lw=2, zorder=1)
ax.scatter(p_ptb, yy, s=64, color=DATASET_COLORS['PTB-XL'], edgecolor='white', linewidth=2, zorder=3, label='PTB-XL (source)')
ax.scatter(p_chp, yy, s=64, color=DATASET_COLORS['Chapman'], edgecolor='white', linewidth=2, zorder=3, label='Chapman (target)')
ax.set_xscale('log'); ax.set_yticks(yy); ax.set_yticklabels(CLASSES); ax.invert_yaxis()
_lo, _hi = min(p_ptb.min(), p_chp.min()) / 1.5, max(p_ptb.max(), p_chp.max()) * 1.5
_ticks = [v for v in (0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100) if _lo <= v <= _hi]
ax.set_xlim(_lo, _hi); ax.set_xticks(_ticks); ax.set_xticklabels([f'{v:g}%' for v in _ticks])
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlabel('Prevalence (% of records, log scale)')
ax.set_title(f'Label shift between datasets ({CFG.label_set})')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2)
savefig(fig, 'fig02_label_prevalence_shift')

---
# Cell 3 — Preprocessing & PyTorch Datasets 🧹

## 3.1 LOCKED preprocessing protocol (dono datasets + saare models ke liye identical)

| Step | PTB-XL | Chapman | Kyun |
|---|---|---|---|
| 1. Units | mV (wfdb) | `(val − baseline) / gain` → mV (gain header se, 1000/mV) | Same physical scale |
| 2. Lead order | naam se verify → `I, II, III, aVR, aVL, aVF, V1–V6` | same | Lead *i* ka matlab dono datasets mein same hona chahiye |
| 3. Sampling | `records100/` (already 100 Hz) | 500 → 100 Hz with `scipy.signal.resample` | Input shape `[B, 12, 1000]` |
| 4. Bandpass | 0.5–40 Hz, Butterworth order 4, zero-phase | same | Baseline wander (< 0.5 Hz) aur muscle/HF noise (> 40 Hz) hatana |
| 5. Normalize | per-record, per-lead z-score | same | Amplitude/offset scale ke differences hatana |

## 3.2 Resampling: `scipy.signal.resample` (FFT method) + Gibbs fix

FFT resampling signal ko frequency domain mein le jaata hai, naye Nyquist (50 Hz) se upar ke components kaat deta hai (built-in anti-aliasing ✅), aur phir wapas time domain mein laata hai.

**Problem:** FFT maanta hai ki signal *periodic* hai. Agar 10 s ECG ka start aur end value alag hai (baseline wander ki wajah se), to edges pe artificial "ringing" aa jaati hai (Gibbs phenomenon).
**Fix:** resample se pehle start aur end ko jodne wali seedhi line (ramp) subtract karo, aur resample ke baad wapas add kar do. Ye simple aur exact hai, aur Grad-CAM ko edge artifacts pe focus karne se bhi bachata hai.

> Optional: `CFG.ptbxl_from_500hz = True` karoge to PTB-XL bhi 500 Hz files se *same* resampler se 100 Hz banega, yaani resampling pipeline ki bhi full parity (loading 5× slow).

## 3.3 Bandpass filter: Butterworth + SOS + filtfilt

- **Butterworth** filter ka passband maximally flat hota hai (koi ripple nahi), isliye ECG wave shapes distort nahi hote.
- **SOS (second-order sections):** 100 Hz pe 0.5 Hz cutoff normalized frequency 0.01 hai. Itne chhote cutoff pe classic `(b, a)` coefficients numerically unstable ho jaate hain; SOS form stable rehti hai.
- **Zero-phase (`sosfiltfilt`):** filter ek baar aage aur ek baar peeche chalta hai, isliye phase shift cancel ho jaata hai aur QRS/T-wave ki **timing nahi badalti** (Grad-CAM alignment ke liye critical). Side-effect: effective magnitude response |H(f)|² ho jaata hai (zyada steep).

## 3.4 Z-score + flat leads

`z = (x − mean) / (std + 1e-8)` har record ki har lead pe alag lagta hai. Agar koi lead lagbhag flat hai (std < 1e-4 mV, matlab electrode disconnected), to ye formula noise ko ~10⁴× amplify kar deta. Isliye aisi lead ko **0** set karte hain aur count report karte hain. Jis record ki saari 12 leads flat hon, woh drop ho jaata hai.

## 3.5 Caching + memory

Preprocessed arrays `.npy` files mein save hote hain (`physiodks_cache/`). PTB-XL ≈ 1.05 GB aur Chapman ≈ 2.2 GB (float32). Chapman ko **memory-map** se padha jaata hai, poora RAM mein load nahi hota. Files chunks mein likhi jaati hain, isliye peak RAM bhi kam rehti hai.

### 📖 Glossary: Cell 3

| Term | Hinglish explanation |
|---|---|
| **Sampling rate (fs)** | 1 second mein kitne samples. 100 Hz ⇒ har 10 ms pe ek sample. |
| **Nyquist frequency** | fs/2. Isse upar ki frequencies represent nahi ho sakti (100 Hz pe 50 Hz). |
| **Aliasing** | Bina filter ke downsample karne par high frequency ka galti se low frequency jaisa dikhna. |
| **Resampling** | Sampling rate badalna (500 → 100 Hz = 5000 → 1000 samples). |
| **FFT** | Fast Fourier Transform: signal ko alag-alag frequencies mein todna. |
| **Gibbs phenomenon / ringing** | Sharp jump ke paas FFT-based processing se aane wali artificial oscillations. |
| **Baseline wander** | Saans ya movement se ECG ki baseline ka dheere-dheere upar-neeche hona (< 0.5 Hz). |
| **Bandpass filter** | Sirf ek frequency range (0.5–40 Hz) ko pass karta hai. |
| **Butterworth filter** | Classic filter design jiska passband flat hota hai. Order zyada ⇒ cutoff zyada sharp. |
| **Zero-phase filtering (filtfilt)** | Forward + backward filtering, jisse waves ki timing shift nahi hoti. |
| **SOS** | Filter ko chhote 2nd-order blocks mein todna; numerically stable. |
| **Z-score normalization** | Data ko mean 0 aur standard deviation 1 banana. |
| **Flat lead** | Lead jisme koi signal nahi (disconnected electrode); lagbhag constant value. |
| **PSD (Power Spectral Density, Welch)** | Kaunsi frequency pe kitni energy hai. Domain shift dekhne ka ek tareeka. |
| **Memory-map (memmap)** | File ko RAM mein load kiye bina, zaroorat ke hisaab se disk se padhna. |
| **`Dataset` / `DataLoader`** | PyTorch classes: `Dataset` batata hai "index *i* pe kaunsa sample hai"; `DataLoader` samples ko batches mein jodta hai aur shuffle karta hai. |
| **Batch / tensor shape `[B, 12, 1000]`** | B ECGs ek saath; 12 leads (channels); 1000 time-steps. |

In [ ]:
# =====================================================================
# Cell 3.1: Signal-processing primitives (dono datasets ke liye IDENTICAL)
# =====================================================================
SOS = butter(CFG.filter_order, CFG.bandpass_hz, btype='bandpass', fs=CFG.fs, output='sos')   # order-4 Butterworth (SOS)


def resample_fft(sig: np.ndarray, n_out: int) -> np.ndarray:
    """[L, T_in] -> [L, n_out] using scipy.signal.resample (FFT method) + endpoint-ramp Gibbs fix."""
    T = sig.shape[-1]
    a, b = sig[..., :1], sig[..., -1:]                                  # start & end values [L, 1]
    ramp_in = np.arange(T) / (T - 1)                                    # 0 ... 1
    detrended = sig - (a + (b - a) * ramp_in)                           # ab start = end = 0 (smooth periodic extension)
    out = resample(detrended, n_out, axis=-1)
    ramp_out = np.arange(n_out) * (T / n_out) / (T - 1)                 # same physical time points
    return (out + (a + (b - a) * ramp_out)).astype(np.float32)


def bandpass(x: np.ndarray) -> np.ndarray:
    """Zero-phase Butterworth bandpass along last (time) axis. x: [..., T]."""
    return sosfiltfilt(SOS, x, axis=-1).astype(np.float32)


def zscore_leads(x: np.ndarray, flat_std: float) -> Tuple[np.ndarray, np.ndarray]:
    """Per-record, per-lead z-score (x - mean) / (std + 1e-8). x: [N, 12, T] -> (z [N, 12, T], flat-mask [N, 12])."""
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True)
    z = (x - mu) / (sd + 1e-8)
    flat = sd[..., 0] < flat_std                                        # [N, 12]
    z[flat] = 0.0                                                       # flat/missing lead => zeros
    return z.astype(np.float32), flat


def reorder_to_standard(sig: np.ndarray, names: Sequence[str]) -> np.ndarray:
    """Leads ko standard order [I, II, III, aVR, aVL, aVF, V1..V6] mein lao (case-insensitive name match)."""
    upper = [str(n).strip().upper() for n in names]
    missing = [l for l in LEADS if l.upper() not in upper]
    if missing:
        raise ValueError(f'Leads missing {missing} in {list(names)}')
    return sig[[upper.index(l.upper()) for l in LEADS]]


# ---------------- Sanity check 1: filter frequency response ----------------
f_hz, h = sos_freqz(SOS, worN=4096, fs=CFG.fs)
mag2_db = 20 * np.log10(np.maximum(np.abs(h) ** 2, 1e-12))             # filtfilt => |H|^2
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.axvspan(*CFG.bandpass_hz, color=GRID, alpha=0.6, lw=0)
ax.plot(f_hz, mag2_db, color=MODEL_COLORS['PhysioDKS-IncepSE'])
for fc in CFG.bandpass_hz:
    ax.axvline(fc, color=AXIS, lw=0.8)
ax.set_ylim(-80, 5); ax.set_xlim(0, CFG.fs / 2)
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Gain (dB)')
ax.set_title(f'Zero-phase Butterworth bandpass {CFG.bandpass_hz} Hz, order {CFG.filter_order} (effective |H|², fs={CFG.fs} Hz)')
savefig(fig, 'fig03a_bandpass_response')

# ---------------- Sanity check 2: synthetic tones ----------------
_t = np.arange(CFG.n_samples) / CFG.fs
_tones = {'0.15 Hz baseline wander': 0.15, '1.2 Hz (T-wave band)': 1.2, '10 Hz (QRS band)': 10.0, '45 Hz (HF/muscle noise)': 45.0}
_gains = {}
for _name, _f0 in _tones.items():
    _x = np.sin(2 * np.pi * _f0 * _t)
    _y = bandpass(_x[None])[0]
    _gains[_name] = np.std(_y[100:-100]) / np.std(_x[100:-100])         # edges chhod ke amplitude gain
    print(f'{_name:26s} -> amplitude gain {_gains[_name]:.4f}')
assert _gains['10 Hz (QRS band)'] > 0.95 and _gains['1.2 Hz (T-wave band)'] > 0.9, 'Passband attenuate ho raha hai!'
assert _gains['0.15 Hz baseline wander'] < 0.05 and _gains['45 Hz (HF/muscle noise)'] < 0.05, 'Stopband leak!'

# ---------------- Sanity check 3: FFT resample + Gibbs fix ----------------
_t500 = np.arange(5000) / 500
_sig = (np.sin(2 * np.pi * 1.0 * _t500) + 0.8 * _t500 / 10)[None]       # 1 Hz wave + strong drift (start != end)
_plain = resample(_sig, 1000, axis=-1)
_fixed = resample_fft(_sig, 1000)
_truth = _sig[:, ::5]
print(f'Resample edge error (first 20 samples): plain FFT {np.abs(_plain - _truth)[:, :20].max():.4f} | '
      f'with ramp fix {np.abs(_fixed - _truth)[:, :20].max():.4f}')
print('✅ Filter + resampler sanity checks passed')

In [ ]:
# =====================================================================
# Cell 3.2: Load -> (resample) -> bandpass -> z-score -> cache (.npy, chunked)
# =====================================================================
PREP_TAG = (f'fs{CFG.fs}_bp{CFG.bandpass_hz[0]}-{CFG.bandpass_hz[1]}_o{CFG.filter_order}_'
            f'{"ptb500" if CFG.ptbxl_from_500hz else "ptb100"}_{CFG.mode_tag}')
CHUNK = 512


def _finalize_chunk(raw: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """raw [n, 12, T] (mV) -> bandpass -> z-score. Returns (z, flat-lead mask)."""
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    return zscore_leads(bandpass(raw), CFG.flat_lead_std_mv)


def _new_memmap(path: Path, n: int) -> np.memmap:
    return np.lib.format.open_memmap(path, mode='w+', dtype=np.float32, shape=(n, CFG.n_leads, CFG.n_samples))


def preprocess_ptbxl(cfg: Config, db: pd.DataFrame) -> Tuple[np.ndarray, pd.DataFrame]:
    x_file, q_file = cfg.cache_dir / f'ptbxl_X_{PREP_TAG}.npy', cfg.cache_dir / f'ptbxl_quality_{PREP_TAG}.csv'
    if x_file.exists() and q_file.exists():
        q = pd.read_csv(q_file, index_col='ecg_id')
        if q.index.equals(db.index):
            return np.load(x_file), q                                    # PTB-XL poora RAM mein (~1 GB)
        print('⚠️ PTB-XL cache index mismatch -> recompute')
    col = 'filename_hr' if cfg.ptbxl_from_500hz else 'filename_lr'
    tmp = x_file.with_name(x_file.stem + '.tmp.npy')
    X = _new_memmap(tmp, len(db))
    n_flat = np.zeros(len(db), dtype=int)
    finite = np.ones(len(db), dtype=bool)
    files = db[col].values
    for s in tqdm(range(0, len(db), CHUNK), desc='PTB-XL preprocess'):
        chunk = files[s:s + CHUNK]
        raw = np.zeros((len(chunk), cfg.n_leads, cfg.n_samples), dtype=np.float32)
        for j, fn in enumerate(chunk):
            sig, fields = wfdb.rdsamp(str(cfg.ptbxl_path / fn))               # [T_native, 12] in mV
            sig = reorder_to_standard(sig.T.astype(np.float32), fields['sig_name'])
            if sig.shape[-1] != cfg.n_samples:                                 # 500 Hz file => 100 Hz
                sig = resample_fft(sig, cfg.n_samples)
            finite[s + j] = np.isfinite(sig).all()
            raw[j] = sig
        z, flat = _finalize_chunk(raw)
        X[s:s + len(chunk)] = z
        n_flat[s:s + len(chunk)] = flat.sum(1)
    X.flush()
    del X                                                                      # file handle band (Windows rename ke liye)
    tmp.replace(x_file)                                                        # half-written cache kabhi use na ho
    q = pd.DataFrame({'ok': finite & (n_flat < cfg.n_leads), 'n_flat_leads': n_flat}, index=db.index)
    q.to_csv(q_file)
    return np.load(x_file), q


def load_chapman_signal(cfg: Config, row) -> np.ndarray:
    """Ek Chapman record: .mat ('val' int16 [12, 5000]) -> mV -> standard lead order -> 100 Hz [12, 1000]."""
    val = scipy.io.loadmat(str(cfg.chapman_path / (row.rel_path + '.mat')))['val'].astype(np.float32)
    gains = np.array(str(row.gains).split(';'), dtype=np.float32)[:, None]
    base = np.array(str(row.baselines).split(';'), dtype=np.float32)[:, None]
    sig = reorder_to_standard((val - base) / gains, str(row.leads).split(';'))     # digital -> mV
    n_native = int(round(sig.shape[-1] * cfg.fs / float(row.fs)))                   # 5000 @ 500 Hz -> 1000 @ 100 Hz
    sig = resample_fft(sig, n_native)
    if n_native < cfg.n_samples:                                                    # chhota record => zero-pad
        sig = np.pad(sig, ((0, 0), (0, cfg.n_samples - n_native)))
    return sig[:, :cfg.n_samples]


def preprocess_chapman(cfg: Config, cdf: pd.DataFrame) -> Tuple[np.ndarray, pd.DataFrame]:
    x_file, q_file = cfg.cache_dir / f'chapman_X_{PREP_TAG}.npy', cfg.cache_dir / f'chapman_quality_{PREP_TAG}.csv'
    if x_file.exists() and q_file.exists():
        q = pd.read_csv(q_file, dtype={'record': str}, keep_default_na=False)
        if q.record.tolist() == cdf.record.astype(str).tolist():
            return np.load(x_file, mmap_mode='r'), q                         # Chapman: memory-map (RAM bachao)
        print('⚠️ Chapman cache mismatch -> recompute')
    tmp = x_file.with_name(x_file.stem + '.tmp.npy')
    X = _new_memmap(tmp, len(cdf))
    n_flat = np.zeros(len(cdf), dtype=int)
    reason = np.array([''] * len(cdf), dtype=object)
    rows = list(cdf.itertuples(index=False))
    for s in tqdm(range(0, len(rows), CHUNK), desc='Chapman preprocess'):
        chunk = rows[s:s + CHUNK]
        raw = np.zeros((len(chunk), cfg.n_leads, cfg.n_samples), dtype=np.float32)
        for j, row in enumerate(chunk):
            try:
                sig = load_chapman_signal(cfg, row)
                if not np.isfinite(sig).all():
                    raise ValueError('non-finite')
                raw[j] = sig
            except Exception as e:                                           # corrupt/missing file => flag, skip
                reason[s + j] = f'{type(e).__name__}: {str(e)[:60]}'
        z, flat = _finalize_chunk(raw)
        X[s:s + len(chunk)] = z
        n_flat[s:s + len(chunk)] = flat.sum(1)
    X.flush()
    del X
    tmp.replace(x_file)
    q = pd.DataFrame({'record': cdf.record.astype(str).values, 'ok': (reason == '') & (n_flat < cfg.n_leads),
                      'n_flat_leads': n_flat, 'fail_reason': reason})
    q.to_csv(q_file, index=False)
    return np.load(x_file, mmap_mode='r'), q


_t0 = time.time()
X_ptb, q_ptb = preprocess_ptbxl(CFG, ptb_db)            # [N_ptb, 12, 1000] float32 (RAM)
X_chap, q_chap = preprocess_chapman(CFG, chap_df)       # [N_chap, 12, 1000] float32 (memmap)
print(f'Preprocessing/cache load: {time.time() - _t0:.1f}s | PTB-XL {X_ptb.shape} | Chapman {X_chap.shape}')

# ---------- Quality filter -> FINAL indices ----------
ok_ptb = q_ptb['ok'].values.astype(bool)
ok_chap = q_chap['ok'].values.astype(bool)
idx_train, idx_val, idx_test = idx_train[ok_ptb[idx_train]], idx_val[ok_ptb[idx_val]], idx_test[ok_ptb[idx_test]]
idx_ext = np.where(keep_chap & ok_chap)[0]
print(f'PTB-XL : dropped {int((~ok_ptb).sum())} bad records | records with >=1 flat lead: {int((q_ptb.n_flat_leads > 0).sum())}')
print(f'Chapman: dropped {int((~ok_chap).sum())} bad records | records with >=1 flat lead: {int((q_chap.n_flat_leads > 0).sum())}')
if (q_chap.fail_reason != '').any():
    print('Chapman load failures (top):', q_chap.loc[q_chap.fail_reason != '', 'fail_reason'].value_counts().head(3).to_dict())
print(f'FINAL -> train {len(idx_train):,} | val {len(idx_val):,} | test {len(idx_test):,} | Chapman ext {len(idx_ext):,}')

In [ ]:
# =====================================================================
# Cell 3.3: PyTorch Dataset + DataLoader
# =====================================================================
class ECGDataset(Dataset):
    """(signal [12, 1000], multi-hot label [C]) pairs. X numpy array ya memmap dono chalega."""

    def __init__(self, X: np.ndarray, Y: np.ndarray, indices: np.ndarray):
        self.X, self.Y, self.idx = X, Y.astype(np.float32), np.asarray(indices)

    def __len__(self) -> int:
        return len(self.idx)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, torch.Tensor]:
        j = self.idx[i]
        x = torch.from_numpy(np.array(self.X[j], dtype=np.float32))   # [12, 1000] (copy => memmap-safe)
        y = torch.from_numpy(self.Y[j])                               # [C]
        return x, y


def make_loader(X: np.ndarray, Y: np.ndarray, idx: np.ndarray, batch_size: int, shuffle: bool,
                seed: int = 0, drop_last: bool = False) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(seed)                                               # deterministic shuffling
    return DataLoader(ECGDataset(X, Y, idx), batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,
                      num_workers=CFG.num_workers, pin_memory=(DEVICE.type == 'cuda'), generator=g)


# ---------- Shape + normalization sanity checks ----------
_xb, _yb = next(iter(make_loader(X_ptb, Y_ptb, idx_train[:8], batch_size=8, shuffle=False)))
print('Batch x:', tuple(_xb.shape), '# [B, 12, 1000] | Batch y:', tuple(_yb.shape), f'# [B, {N_CLASSES}]')
assert _xb.shape == (8, CFG.n_leads, CFG.n_samples) and _yb.shape == (8, N_CLASSES)
for _name, _X, _idx in [('PTB-XL train', X_ptb, idx_train), ('Chapman ext', X_chap, idx_ext)]:
    _s = np.array(_X[np.sort(np.random.default_rng(0).choice(_idx, min(200, len(_idx)), replace=False))])
    _live = _s.std(-1) > 0                                            # flat (zeroed) leads exclude
    print(f'{_name:12s}: per-lead mean {np.abs(_s.mean(-1))[_live].max():.2e} (≈0) | '
          f'std {_s.std(-1)[_live].mean():.4f} (≈1) | flat leads {int((~_live).sum())}')

In [ ]:
# =====================================================================
# Cell 3.4: Visual sanity check + domain-shift diagnostic (PSD)
# =====================================================================
# (a) Raw vs processed, lead II, ek-ek record dono datasets se
_ptb_i, _chp_i = idx_test[0], idx_ext[0]
_raw_ptb, _fields = wfdb.rdsamp(str(CFG.ptbxl_path / ptb_db[('filename_hr' if CFG.ptbxl_from_500hz else 'filename_lr')].iloc[_ptb_i]))
_raw_ptb = reorder_to_standard(_raw_ptb.T, _fields['sig_name'])
if _raw_ptb.shape[-1] != CFG.n_samples:
    _raw_ptb = resample_fft(_raw_ptb, CFG.n_samples)
_raw_chp = load_chapman_signal(CFG, chap_df.iloc[_chp_i])
_tt = np.arange(CFG.n_samples) / CFG.fs
fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharex=True)
for r, (name, raw, proc) in enumerate([('PTB-XL', _raw_ptb, X_ptb[_ptb_i]), ('Chapman', _raw_chp, X_chap[_chp_i])]):
    axes[r, 0].plot(_tt, raw[1], color=DATASET_COLORS[name], lw=0.8)
    axes[r, 0].set_title(f'{name}: raw lead II (mV, 100 Hz)')
    axes[r, 0].set_ylabel('mV')
    axes[r, 1].plot(_tt, proc[1], color=DATASET_COLORS[name], lw=0.8)
    axes[r, 1].set_title(f'{name}: bandpass + z-score (model input)')
    axes[r, 1].set_ylabel('z')
for ax in axes[1]:
    ax.set_xlabel('Time (s)')
savefig(fig, 'fig03b_raw_vs_processed')

# (b) Median PSD of lead II after preprocessing: residual acquisition shift
def median_psd(X: np.ndarray, idx: np.ndarray, n: int = 500, lead: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    sel = np.sort(np.random.default_rng(1).choice(idx, min(n, len(idx)), replace=False))
    fr, P = welch(np.array(X[sel][:, lead]), fs=CFG.fs, nperseg=256, axis=-1)
    return fr, np.median(10 * np.log10(P + 1e-12), axis=0)

fig, ax = plt.subplots(figsize=(8, 3.6))
for name, X_, idx_ in [('PTB-XL', X_ptb, idx_test), ('Chapman', X_chap, idx_ext)]:
    fr, P = median_psd(X_, idx_)
    ax.plot(fr, P, color=DATASET_COLORS[name], label=name)
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Power (dB, z-units²/Hz)')
ax.set_title('Lead II median PSD after identical preprocessing (gap = residual domain shift)')
ax.legend(loc='upper right')
savefig(fig, 'fig03c_psd_domain_shift')

---
# Cell 4 — Model Architectures 🧠

Saare 4 models **same input** `[B, 12, 1000]` lete hain aur **same head** (GAP → Dropout → Linear) se `[B, C]` logits dete hain. Sirf "body" alag hai, isliye comparison fair rehta hai.

| # | Model | Role in study | Key idea |
|---|---|---|---|
| 1 | **ResNet1D** | Baseline | Standard 1D residual CNN (stride se downsampling), ~0.55M params (InceptionTime ~0.47M ke barabar, taaki capacity confound na kare) |
| 2 | **InceptionTime** | Base architecture | Multi-scale kernels (39/19/9) + residual connections, full temporal resolution |
| 3 | **IncepSE** | Ablation (SE ka effect) | InceptionTime + SE channel attention, 1×1 bottleneck ke turant baad |
| 4 | **PhysioDKS-IncepSE** | **Proposed** | Stage-1 physiological DKS filter + InstanceNorm + Lead-SE → IncepSE backbone |

**Ablation logic:** (2 → 3) = SE ka contribution; (3 → 4) = Stage-1 ka contribution. `RUN_EXTENDED_ABLATION = True` se Stage-1 ke andar ke components (DKS / InstanceNorm / Lead-SE / physiological init) ka effect bhi alag-alag milta hai.

## 4.1 Physiological kernel sizes (100 Hz pe 1 sample = 10 ms)

| Branch | Kernel | Duration | ECG target | Physio-init Ricker center freq |
|---|---|---|---|---|
| A | k = 5 | ~50 ms | sharp R-peak / QRS upstroke | ~28 Hz |
| B | k = 11 | ~110 ms | PR/QRS transitions, poora QRS (80–120 ms) | ~13 Hz |
| C | k = 21 | ~210 ms | smooth P-wave (~80–110 ms) aur T-wave envelopes (~150–250 ms) | ~7 Hz |

**Physiological initialization:** har branch ke `m = 4` filters per lead Gaussian-derivative wavelets se start hote hain (σ = k/6). Order 0 = smoothing, order 1 = slope/edge, order 2 = Ricker "peak detector", order 3 = sharp transitions. Training ke dauran ye learnable hain (fine-tune hote hain). Cell 7 mein dekhenge ki training ke baad bhi branches apne frequency bands mein rahe ya nahi.

## 4.2 PhysioDKS-IncepSE ka data flow

```
x [B, 12, 1000]  (z-scored, band-passed)
 │  ── Stage 1: PhysioDKS Filter (depthwise = har lead ka apna filter) ──────────────────────────────
 ├─ Branch A: DepthwiseConv k=5  ─┐
 ├─ Branch B: DepthwiseConv k=11 ─┼─► U_k [B, 3, 48, 1000]
 └─ Branch C: DepthwiseConv k=21 ─┘        │
                                           ├─ squeeze: log-RMS energy of every U_k → LayerNorm → MLP → softmax over 3 branches
                                           ▼
                        V = Σ_k a_k · U_k   (DKS: input-adaptive filter)               [B, 48, 1000]
                        e = log-RMS(V)  (pre-IN energy descriptor)                      [B, 48]
                        V = InstanceNorm1d(V, affine=True)                              [B, 48, 1000]
                        V = V · LeadGate(LayerNorm(e))  (12 sigmoid lead weights)       [B, 48, 1000]  (lead-grouped)
 │  ── Stage 2: IncepSE backbone (2 residual stages × 3 IncepSE modules) ───────────────────────────
 ▼                                                                                [B, 128, 1000]
 GAP → Dropout(0.3) → Linear                                                      [B, C] logits
```

## 4.3 Teen design decisions (engineering insights jo zyada-tar implementations miss karti hain)

1. **Depthwise (per-lead) Stage-1:** `groups=12` conv mein har lead ka apna filter hota hai aur leads mix nahi hote. Isliye Stage-1 ke output channels **lead-grouped** rehte hain (`channel = lead × m + j`). Isi wajah se hum *lead-resolved* Grad-CAM aur genuinely *lead-wise* SE weights nikaal paate hain. (IncepSE blocks ke andar ka SE "channel" attention hai; wahan channels mixed features hain, leads nahi.)
2. **Energy (RMS) squeeze, GAP nahi:** z-scored, band-passed ECG ka time-average ≈ 0 hota hai. Isliye standard SE/SKNet ka Global-Average-Pool har input ke liye ~0 deta, aur attention **input-independent** (bekaar) ho jaati. DKS aur Lead-SE isliye `sqrt(mean(x²))` (RMS energy) squeeze use karte hain (Cell 4.6 mein numerically dikhaya hai). Raw energies records ke beech sirf kuch % badalti hain (upar se ek bada constant offset), isliye unhe **log-scale** pe lekar har record ke andar **LayerNorm** se standardize karte hain. Isse gating MLP ko *relative* band/lead pattern milta hai ("is record ki energy QRS-band mein zyada hai ya P/T-band mein?"), aur gates sach mein input-dependent bante hain. LayerNorm per-record hai (koi batch statistics nahi), bilkul InstanceNorm ki philosophy ki tarah. Isi reason se IncepSE ka SE default `avgstd` (mean + std "statistics pooling", SRM-style) squeeze use karta hai. Original paper wala `'avg'` bhi config (`CFG.se_squeeze`) se available hai.
3. **InstanceNorm scale-invariant hai:** IN har channel ko uske apne mean/std se normalize karta hai, isliye IN se *pehle* lagaya gaya koi bhi per-channel gain (jaise SE gate) **cancel** ho jaata. Isliye Lead-SE ka descriptor IN se pehle ki energies se banta hai, lekin gate IN ke *baad* apply hota hai.

### 📖 Glossary: Cell 4

| Term | Hinglish explanation |
|---|---|
| **Conv1d** | Ek chhota filter (kernel) jo signal pe slide karta hai aur har position pe weighted sum nikaalta hai. Basically ek pattern detector. |
| **Kernel size** | Filter ek saath kitne samples dekhta hai (k = 21 ⇒ 210 ms). |
| **Receptive field** | Output ka ek point input ke kitne lambe hisse ko "dekh" sakta hai. Layers ke saath badhta hai. |
| **Stride / Padding** | Stride = filter kitne steps kood ke chale (stride 2 ⇒ length aadhi). Padding = edges pe zeros jodna taaki length same rahe. |
| **Channels** | Feature maps ki "depth". Input mein 12 channels = 12 leads. |
| **BatchNorm1d** | Har channel ko *poore batch* ke statistics se normalize karta hai, jisse training stable aur fast hoti hai. Inference pe training ke running stats use hote hain. |
| **InstanceNorm1d** | Har channel ko *har single record* ke apne statistics se normalize karta hai, jisse record-specific amplitude/offset hat jaata hai (domain shift ke against). `affine=True` ⇒ learnable scale (γ) + shift (β). |
| **ReLU** | Activation function `max(0, x)`, jo network ko non-linearity deta hai. |
| **Residual / skip connection** | `output = F(x) + x`. Gradient ko seedha raasta milta hai, isliye deep networks train ho paate hain (ResNet ka core idea). |
| **ResNet** | Residual blocks se bana CNN. |
| **Inception module** | Ek hi layer mein multiple kernel sizes parallel (39/19/9) + ek max-pool branch, phir concat ⇒ multi-scale features. |
| **Bottleneck (1×1 conv)** | Channels kam karne wali 1×1 convolution; compute bachata hai. |
| **Max-pool** | Window ka maximum lena; chhote shifts ke against robust. |
| **SE block (Squeeze-and-Excitation)** | *Squeeze*: har channel ko ek number mein summarize karo. *Excitation*: chhota MLP + sigmoid se har channel ka importance weight (0–1) nikaalo. Phir channels ko us weight se multiply karo. |
| **Sigmoid** | `1/(1+e^{-x})`, jo kisi bhi number ko (0, 1) range mein laata hai. |
| **Softmax** | Numbers ko probabilities mein badalta hai jinka sum 1 ho. |
| **Depthwise convolution (`groups`)** | Har input channel (lead) ka apna alag filter; channels aapas mein mix nahi hote. |
| **DKS / Selective-Kernel attention** | Kai kernel sizes ke outputs mein se *har input ke hisaab se* softmax weights se mix choose karna (SKNet idea). |
| **RMS energy** | `sqrt(mean(x²))`, signal ki "taakat". Zero-mean signals ke liye bhi informative rehta hai. |
| **Log-energy** | Energy ka log. Multiplicative farak (gain ×2) ko additive bana deta hai, jisse neural network ke liye seekhna aasaan hota hai. |
| **LayerNorm** | Ek single sample ke saare features ko milakar normalize karna (mean 0, std 1). BatchNorm ki tarah batch pe depend nahi karta. |
| **Gaussian-derivative / Ricker wavelet** | Gaussian bell curve aur uske derivatives: classic filters jo smoothing (order 0), slope (order 1) aur peaks (order 2) detect karte hain. Inse Stage-1 initialize hota hai. |
| **GAP (Global Average Pooling)** | Time axis pe average: `[B, C, T] → [B, C]`. |
| **Dropout** | Training mein random neurons ko 0 karna, jisse overfitting kam hoti hai. |
| **Logits** | Sigmoid se pehle ke raw scores. Probability = sigmoid(logit). |
| **Parameters** | Model ke learnable numbers (weights). Zyada params ⇒ zyada capacity, par overfitting ka risk bhi. |

In [ ]:
# =====================================================================
# Cell 4.1: Shared building blocks
# =====================================================================
def squeeze_stats(x: torch.Tensor, mode: str) -> torch.Tensor:
    """Channel descriptor for attention. x: [B, C, T] -> [B, C] ('avg'/'rms') ya [B, 2C] ('avgstd'). FP32 for AMP safety."""
    xf = x.float()
    if mode == 'avg':                        # original SE paper (Global Average Pooling)
        return xf.mean(-1)
    if mode == 'rms':                        # energy: zero-mean ECG ke liye bhi informative
        return torch.sqrt(xf.pow(2).mean(-1) + 1e-6)
    if mode == 'avgstd':                     # statistics pooling (SRM-style): mean + std
        return torch.cat([xf.mean(-1), torch.sqrt(xf.var(-1, unbiased=False) + 1e-5)], dim=1)
    raise ValueError(mode)


class SqueezeExcite1D(nn.Module):
    """Squeeze-and-Excitation (Hu et al., 2018) for 1D signals: x * sigmoid(MLP(squeeze(x)))."""

    def __init__(self, channels: int, reduction: int = 8, squeeze: str = 'avgstd'):
        super().__init__()
        self.squeeze = squeeze
        in_feats = channels * (2 if squeeze == 'avgstd' else 1)
        hidden = max(channels // reduction, 4)
        self.gate = nn.Sequential(nn.Linear(in_feats, hidden), nn.ReLU(), nn.Linear(hidden, channels), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # x: [B, C, T]
        w = self.gate(squeeze_stats(x, self.squeeze))             # [B, C] in (0, 1)  (XAI: hook on self.gate)
        return x * w.unsqueeze(-1).to(x.dtype)                    # [B, C, T]


class ClassifierHead(nn.Module):
    """GAP -> Dropout -> Linear. Saare 4 models yahi head use karte hain (parity)."""

    def __init__(self, in_ch: int, n_classes: int, p_drop: float = 0.3):
        super().__init__()
        self.drop = nn.Dropout(p_drop)
        self.fc = nn.Linear(in_ch, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # x: [B, C, T]
        return self.fc(self.drop(x.mean(dim=-1)))                 # [B, n_classes] logits


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# =====================================================================
# Cell 4.2: Model 1 (Baseline): ResNet1D
# =====================================================================
class BasicBlock1D(nn.Module):
    """conv-BN-ReLU-conv-BN + identity/projection shortcut, then ReLU."""

    def __init__(self, in_ch: int, out_ch: int, k: int = 7, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, k, stride=stride, padding=k // 2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, padding=k // 2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.shortcut = nn.Identity() if (stride == 1 and in_ch == out_ch) else nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm1d(out_ch))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNet1D(nn.Module):
    """Standard 1D ResNet (3 stages x 2 BasicBlocks, k=7). [B,12,1000] -> stem s2 -> [B,32,500] -> stages -> [B,128,125].
    Widths chosen for parameter parity with InceptionTime (~0.5M params) so capacity doesn't confound the comparison."""

    def __init__(self, n_leads: int = 12, n_classes: int = 9, widths: Sequence[int] = (32, 64, 128),
                 strides: Sequence[int] = (1, 2, 2), k: int = 7, p_drop: float = 0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(n_leads, widths[0], 15, stride=2, padding=7, bias=False),
                                  nn.BatchNorm1d(widths[0]), nn.ReLU())
        stages, in_ch = [], widths[0]
        for w, s in zip(widths, strides):
            stages.append(nn.Sequential(BasicBlock1D(in_ch, w, k, s), BasicBlock1D(w, w, k, 1)))
            in_ch = w
        self.stages = nn.Sequential(*stages)
        self.head = ClassifierHead(in_ch, n_classes, p_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # [B, 12, 1000]
        x = self.stem(x)                                          # [B, 32, 500]
        x = self.stages(x)                                        # [B, 128, 125]
        return self.head(x)                                       # [B, C]

    def gradcam_targets(self) -> Dict[str, nn.Module]:
        return {'final': self.stages[-1]}

In [ ]:
# =====================================================================
# Cell 4.3: Model 2 & 3: InceptionTime / IncepSE
# =====================================================================
class InceptionModule1D(nn.Module):
    """1x1 bottleneck -> [SE] -> parallel convs (k = 39/19/9) + maxpool branch -> concat -> BN -> ReLU."""

    def __init__(self, in_ch: int, nf: int = 32, bottleneck: int = 32, kernel_sizes: Sequence[int] = (39, 19, 9),
                 use_se: bool = False, se_reduction: int = 8, se_squeeze: str = 'avgstd'):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_ch, bottleneck, 1, bias=False)
        self.se = SqueezeExcite1D(bottleneck, se_reduction, se_squeeze) if use_se else nn.Identity()
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck, nf, k, padding=k // 2, bias=False) for k in kernel_sizes])
        self.pool_branch = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_ch, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * (len(kernel_sizes) + 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # x: [B, C_in, T]
        z = self.se(self.bottleneck(x))                           # [B, 32, T]  (IncepSE: SE after bottleneck)
        out = torch.cat([conv(z) for conv in self.convs] + [self.pool_branch(x)], dim=1)   # [B, 128, T]
        return F.relu(self.bn(out))


class InceptionResidualStage(nn.Module):
    """3 Inception modules + residual shortcut: out = ReLU(modules(x) + shortcut(x))."""

    def __init__(self, in_ch: int, nf: int = 32, n_modules: int = 3, **module_kw):
        super().__init__()
        out_ch = nf * 4
        self.mods = nn.Sequential(*[InceptionModule1D(in_ch if i == 0 else out_ch, nf, **module_kw)
                                    for i in range(n_modules)])
        self.shortcut = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # [B, C_in, T] -> [B, 128, T]
        return F.relu(self.mods(x) + self.shortcut(x))


class InceptionBackbone(nn.Module):
    """depth Inception modules = depth/3 residual stages. Temporal length kabhi kam nahi hoti (full resolution)."""

    def __init__(self, in_ch: int, nf: int = 32, depth: int = 6, use_se: bool = False, **module_kw):
        super().__init__()
        assert depth % 3 == 0, 'depth 3 ka multiple hona chahiye'
        stages, c = [], in_ch
        for _ in range(depth // 3):
            stages.append(InceptionResidualStage(c, nf, 3, use_se=use_se, **module_kw))
            c = nf * 4
        self.stages = nn.Sequential(*stages)
        self.out_channels = c

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.stages(x)


class InceptionTime1D(nn.Module):
    """Model 2 (use_se=False): InceptionTime. Model 3 (use_se=True): IncepSE."""

    def __init__(self, n_leads: int = 12, n_classes: int = 9, nf: int = 32, depth: int = 6,
                 use_se: bool = False, p_drop: float = 0.3, **module_kw):
        super().__init__()
        self.backbone = InceptionBackbone(n_leads, nf, depth, use_se, **module_kw)
        self.head = ClassifierHead(self.backbone.out_channels, n_classes, p_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # [B, 12, 1000]
        return self.head(self.backbone(x))                        # backbone: [B, 128, 1000] -> head: [B, C]

    def gradcam_targets(self) -> Dict[str, nn.Module]:
        return {'final': self.backbone.stages[-1]}

In [ ]:
# =====================================================================
# Cell 4.4: Model 4, Stage 1: Physiological Multi-Scale DKS Filter
# =====================================================================
def gaussian_derivative_bank(k: int, m: int) -> np.ndarray:
    """m filters of length k: Gaussian derivatives (Hermite functions) of order 0..m-1, sigma = k/6, unit L2 norm."""
    sigma = k / 6.0
    u = (np.arange(k) - (k - 1) / 2) / sigma
    g = np.exp(-0.5 * u ** 2)
    hermite = [np.ones_like(u), u, u ** 2 - 1, u ** 3 - 3 * u, u ** 4 - 6 * u ** 2 + 3]
    bank = []
    for j in range(m):
        order = j % len(hermite)
        h = hermite[order] * g
        if order > 0:
            h = h - h.mean()                                      # derivatives: exact zero DC gain
        bank.append(h / (np.linalg.norm(h) + 1e-12))
    return np.stack(bank).astype(np.float32)                      # [m, k]


class PhysioDKSFilter(nn.Module):
    """Stage 1 of PhysioDKS-IncepSE.  [B, 12, T] -> [B, 12*m, T] (lead-grouped channels: c = lead*m + j).

    1. Three depthwise (per-lead) conv branches, k = 5 / 11 / 21 (QRS / transitions / P-T waves at 100 Hz).
    2. DKS: per-channel softmax attention over branches. Descriptor = log-RMS energy of every branch x channel
       (GAP of zero-mean ECG ≈ 0), standardized per record with LayerNorm so the gate sees the RELATIVE band pattern.
    3. InstanceNorm1d(affine=True): per-record, per-channel normalization (cross-hospital shift suppression).
    4. Lead-SE: 12 interpretable lead gates; log-energy descriptor from pre-IN features (+ LayerNorm), gate AFTER IN.
    """

    def __init__(self, n_leads: int = 12, m: int = 4, kernel_sizes: Sequence[int] = (5, 11, 21), reduction: int = 2,
                 physio_init: bool = True, use_dks: bool = True, use_instnorm: bool = True, use_lead_se: bool = True):
        super().__init__()
        self.n_leads, self.m, self.K = n_leads, m, len(kernel_sizes)
        self.kernel_sizes = tuple(kernel_sizes)
        C = n_leads * m
        self.out_channels = C
        self.branches = nn.ModuleList([nn.Conv1d(n_leads, C, k, padding=k // 2, groups=n_leads, bias=False)
                                       for k in kernel_sizes])
        if physio_init:
            with torch.no_grad():
                for conv, k in zip(self.branches, kernel_sizes):
                    bank = torch.from_numpy(gaussian_derivative_bank(k, m))           # [m, k]
                    conv.weight.copy_(bank.repeat(n_leads, 1).unsqueeze(1))         # [12*m, 1, k] lead-major
        self.use_dks, self.use_lead_se = use_dks, use_lead_se
        if use_dks:                                               # modules sirf tab banao jab use hon (no dead params)
            d = max(C // reduction, 16)
            self.dks_norm = nn.LayerNorm(self.K * C)              # per-record descriptor standardization
            self.dks_mlp = nn.Sequential(nn.Linear(self.K * C, d), nn.ReLU(), nn.Linear(d, self.K * C))
            nn.init.zeros_(self.dks_mlp[-1].weight)               # start = equal mix (1/3 each) => stable
            nn.init.zeros_(self.dks_mlp[-1].bias)
            self.dks_softmax = nn.Softmax(dim=1)                  # alag module => XAI forward-hook laga sakte hain
        self.inorm = nn.InstanceNorm1d(C, affine=True) if use_instnorm else nn.Identity()
        if use_lead_se:
            self.lead_norm = nn.LayerNorm(C)
            self.lead_gate = nn.Sequential(nn.Linear(C, max(C // 2, 12)), nn.ReLU(),
                                           nn.Linear(max(C // 2, 12), n_leads), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # x: [B, 12, T]
        B, T = x.shape[0], x.shape[-1]
        U = torch.stack([conv(x) for conv in self.branches], dim=1)                   # [B, K, C, T]
        if self.use_dks:
            s = 0.5 * torch.log(U.float().pow(2).mean(-1) + 1e-6)                     # [B, K, C] log-RMS energy per band
            s = self.dks_norm(s.flatten(1))                                           # [B, K*C] per-record standardized
            a = self.dks_softmax(self.dks_mlp(s).float().view(B, self.K, -1))         # [B, K, C], sum_k = 1
        else:
            a = torch.full((B, self.K, self.out_channels), 1.0 / self.K, device=x.device)
        V = (U * a.unsqueeze(-1).to(U.dtype)).sum(1)                                  # [B, C, T]
        e = 0.5 * torch.log(V.float().pow(2).mean(-1) + 1e-6)                         # [B, C] pre-IN log-energies
        V = self.inorm(V)                                                             # [B, C, T]
        if self.use_lead_se:
            w = self.lead_gate(self.lead_norm(e))                                     # [B, 12] lead attention
            V = V * w.repeat_interleave(self.m, dim=1).unsqueeze(-1).to(V.dtype)      # gate AFTER IN
        return V                                                                      # [B, C, T]

In [ ]:
# =====================================================================
# Cell 4.5: Model 4: PhysioDKS-IncepSE (proposed) + model registry
# =====================================================================
class PhysioDKSIncepSE(nn.Module):
    """Stage 1 (PhysioDKSFilter) -> Stage 2 (IncepSE backbone) -> GAP -> Dropout -> Linear."""

    def __init__(self, n_leads: int = 12, n_classes: int = 9, m: int = 4, nf: int = 32, depth: int = 6,
                 p_drop: float = 0.3, stage1_kw: Optional[dict] = None, **module_kw):
        super().__init__()
        self.stage1 = PhysioDKSFilter(n_leads, m, **(stage1_kw or {}))
        self.backbone = InceptionBackbone(self.stage1.out_channels, nf, depth, use_se=True, **module_kw)
        self.head = ClassifierHead(self.backbone.out_channels, n_classes, p_drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:          # [B, 12, 1000]
        h = self.stage1(x)                                        # [B, 48, 1000]  lead-grouped
        h = self.backbone(h)                                      # [B, 128, 1000]
        return self.head(h)                                       # [B, C]

    def gradcam_targets(self) -> Dict[str, nn.Module]:
        return {'stage1': self.stage1, 'final': self.backbone.stages[-1]}


def _physio(stage1_kw: Optional[dict] = None):
    return lambda C, cfg: PhysioDKSIncepSE(cfg.n_leads, C, m=cfg.dks_filters_per_lead, p_drop=cfg.dropout,
                                           stage1_kw=stage1_kw, se_squeeze=cfg.se_squeeze)


MODEL_REGISTRY = {
    'ResNet1D':           lambda C, cfg: ResNet1D(cfg.n_leads, C, p_drop=cfg.dropout),
    'InceptionTime':      lambda C, cfg: InceptionTime1D(cfg.n_leads, C, use_se=False, p_drop=cfg.dropout),
    'IncepSE':            lambda C, cfg: InceptionTime1D(cfg.n_leads, C, use_se=True, p_drop=cfg.dropout,
                                                         se_squeeze=cfg.se_squeeze),
    'PhysioDKS-IncepSE':  _physio(),
    # ---- extended Stage-1 ablations (RUN_EXTENDED_ABLATION = True) ----
    'PhysioDKS-noDKS':    _physio({'use_dks': False}),         # fixed equal mix of 3 branches
    'PhysioDKS-noIN':     _physio({'use_instnorm': False}),    # InstanceNorm hatao
    'PhysioDKS-noLeadSE': _physio({'use_lead_se': False}),     # lead gates hatao
    'PhysioDKS-randInit': _physio({'physio_init': False}),     # random (Kaiming) init instead of wavelets
}


def build_model(name: str, n_classes: int, cfg: Config = CFG) -> nn.Module:
    return MODEL_REGISTRY[name](n_classes, cfg)

In [ ]:
# =====================================================================
# Cell 4.6: Dummy-tensor sanity checks (training se PEHLE har model verify)
# =====================================================================
torch.manual_seed(0)
dummy_x = torch.randn(4, CFG.n_leads, CFG.n_samples)                  # [B=4, 12, 1000]
dummy_y = torch.randint(0, 2, (4, N_CLASSES)).float()                  # [4, C]

# ---- (a) Forward + backward + Grad-CAM target shapes for every model ----
rows = []
for name in CFG.models:
    torch.manual_seed(0)
    model = build_model(name, N_CLASSES).train()
    acts = {}
    hooks = [mod.register_forward_hook(lambda m_, i_, o_, k=key: acts.__setitem__(k, tuple(o_.shape)))
             for key, mod in model.gradcam_targets().items()]
    t0 = time.time()
    logits = model(dummy_x)                                            # [4, C]
    fwd_ms = 1000 * (time.time() - t0)
    assert logits.shape == (4, N_CLASSES), f'{name}: wrong output shape {tuple(logits.shape)}'
    F.binary_cross_entropy_with_logits(logits, dummy_y).backward()
    no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
    assert not no_grad, f'{name}: gradient nahi aaya -> {no_grad[:3]}'
    for h in hooks:
        h.remove()
    rows.append({'model': name, 'params': count_params(model), 'output': tuple(logits.shape),
                 'grad-cam targets': acts, 'cpu fwd (ms, B=4)': round(fwd_ms, 1)})
show(pd.DataFrame(rows).set_index('model'))

# ---- (b) Stage-1 internal checks ----
s1 = PhysioDKSFilter(CFG.n_leads, CFG.dks_filters_per_lead).eval()
cap = {}
_h1 = s1.dks_softmax.register_forward_hook(lambda m_, i_, o_: cap.__setitem__('a', o_))
_h2 = s1.lead_gate.register_forward_hook(lambda m_, i_, o_: cap.__setitem__('w', o_))
with torch.no_grad():
    s1_out = s1(dummy_x)
_h1.remove(); _h2.remove()
M = CFG.dks_filters_per_lead
assert s1_out.shape == (4, CFG.n_leads * M, CFG.n_samples)
assert torch.allclose(cap['a'].sum(1), torch.ones_like(cap['a'].sum(1)), atol=1e-5), 'DKS softmax sum != 1'
assert cap['w'].shape == (4, CFG.n_leads) and bool(((cap['w'] > 0) & (cap['w'] < 1)).all())
print(f'Stage-1 output {tuple(s1_out.shape)} | DKS attention {tuple(cap["a"].shape)} (sum over branches = 1) | '
      f'Lead-SE gates {tuple(cap["w"].shape)} in (0, 1)')

# ---- (c) Lead-grouping check: sirf lead 3 badlo -> sirf channels 3m..4m-1 badlne chahiye ----
x2 = dummy_x.clone()
x2[:, 3] += torch.randn_like(x2[:, 3])
with torch.no_grad():
    diff = (s1.branches[0](x2) - s1.branches[0](dummy_x)).abs().amax(dim=(0, 2))     # [12*m]
changed = torch.nonzero(diff > 1e-6).flatten().tolist()
assert changed == list(range(3 * M, 4 * M)), f'Lead grouping broken: {changed}'
print(f'✅ Lead-grouping verified: lead 3 perturb -> sirf channels {changed} change hue (lead-resolved XAI valid)')

# ---- (d) Kyun RMS squeeze? z-scored ECG pe GAP ≈ 0 (attention input-independent ho jaati) ----
zx = (dummy_x - dummy_x.mean(-1, keepdim=True)) / dummy_x.std(-1, keepdim=True)
with torch.no_grad():
    U1 = s1.branches[1](zx)
    bott = nn.Conv1d(CFG.n_leads, 32, 1, bias=False)(zx)               # IncepSE block-0 bottleneck jaisa
print(f'Stage-1 branch output: |GAP| = {U1.mean(-1).abs().mean():.2e}  vs  RMS = {U1.pow(2).mean(-1).sqrt().mean():.3f}')
print(f'1x1 bottleneck on z-scored input: |GAP| = {bott.mean(-1).abs().max():.2e} (exactly ~0) -> '
      f"isliye SE squeeze = '{CFG.se_squeeze}'")

# ---- (e) Physiological init: har branch ke Ricker (order-2) filter ki center frequency ----
for k in s1.kernel_sizes:
    bank = gaussian_derivative_bank(k, max(M, 3))
    H = np.abs(np.fft.rfft(bank[2], n=1024))
    f = np.fft.rfftfreq(1024, d=1 / CFG.fs)
    print(f'  branch k={k:2d} ({10 * k:3d} ms, sigma={k / 6:.2f} samples): Ricker peak at {f[np.argmax(H)]:5.1f} Hz')
print('✅ All model sanity checks passed')

---
# Cell 5 — Training & Validation Engine 🏋️

## 5.1 LOCKED training protocol ("parity contract": saare models ke liye same)

| Component | Setting | Kyun |
|---|---|---|
| Loss | `BCEWithLogitsLoss` (optional: Focal, ASL) | Multi-label: har class ek independent yes/no sawaal |
| Optimizer | AdamW, weight decay 1e-2 (BN/IN weights aur biases pe 0) | Adaptive step size + decoupled regularization |
| Scheduler | OneCycleLR, `max_lr = 1e-3`, `pct_start = 0.2` | 20% steps warm-up, phir cosine decay: fast + stable convergence |
| Gradient clipping | max-norm 1.0 | Exploding gradients se bachav |
| Precision | AMP (sirf CUDA pe) | ~2× speed, ~aadhi memory |
| Model selection | best **val (fold 9) Macro-AUC** checkpoint | Fold 10 / Chapman kabhi selection mein use nahi hote |
| Early stopping | patience 8 epochs | Overfitting + time dono bachao |
| Seed | 42 (multi-seed optional) | Reproducibility |

## 5.2 Losses for class imbalance

- **BCE:** `−[y·log p + (1−y)·log(1−p)]`, har class ke liye. Ye default hai.
- **Focal (Lin et al., 2017):** BCE × `(1 − p_t)^γ`. Jo examples already easy hain (p_t high), unka loss dab jaata hai aur model hard examples pe focus karta hai.
- **ASL (Ridnik et al., 2021):** positives aur negatives ke liye alag focusing (γ⁻ = 4, γ⁺ = 0) + negative probability shifting (clip 0.05). Multi-label mein bahut saare easy negatives ka dominance kam karta hai.

> ⚠️ Saari losses `mean` reduction use karti hain. `sum` karne se gradient ka magnitude batch size pe depend karta, aur gradient clipping (1.0) ka effect models/losses mein alag ho jaata (unfair comparison).

> **OneCycleLR + early stopping note:** schedule poore `epochs` ke liye bana hai, isliye early stop hone par LR poora anneal nahi hota. Ye standard practice hai, aur saare models pe same rule lagta hai.

> **Loss ablation kaise karein:** `CFG.loss_name = 'focal'` (ya `'asl'`) set karke Cell 5 onwards dobara chalao, aur alag `run_dir` do (jaise `CFG.run_dir = Path('./physiodks_runs_focal')`) taaki BCE ke results overwrite na hon.

### 📖 Glossary: Cell 5

| Term | Hinglish explanation |
|---|---|
| **Loss function** | Prediction kitni galat hai, uska ek number. Training isko minimize karti hai. |
| **BCEWithLogitsLoss** | Sigmoid + binary cross-entropy ek saath (numerically stable). |
| **Focal loss / ASL** | Imbalance ke liye modified BCE (upar dekho). |
| **Optimizer (AdamW)** | Gradients se weights update karne ka algorithm. Adam = adaptive step size; W = weight decay alag se (decoupled). |
| **Weight decay** | Weights ko chhota rakhne ki penalty, jisse overfitting kam hoti hai. |
| **Learning rate (LR)** | Har update ka step size. |
| **OneCycleLR / warm-up** | LR pehle dheere-dheere badhta hai (warm-up), phir dheere-dheere bahut chhota ho jaata hai. |
| **Gradient clipping** | Gradient vector ki length ko max 1.0 tak limit karna. |
| **GradScaler** | AMP mein float16 gradients ko underflow (0 ho jaane) se bachane ke liye loss ko scale karta hai. |
| **Epoch** | Poore training set ka ek pass. |
| **Validation set** | Training ke dauran model select/tune karne ke liye alag rakha gaya data (fold 9). |
| **Early stopping / patience** | Validation metric `patience` epochs tak improve na ho to training rok do. |
| **Checkpoint** | Model weights ki saved file (`best.pt`). |
| **Overfitting** | Model training data ko "yaad" kar leta hai aur naye data pe kharab perform karta hai. |

In [ ]:
# =====================================================================
# Cell 5.1: Losses, optimizer param-groups, AMP helpers, inference
# =====================================================================
class FocalLossWithLogits(nn.Module):
    """Sigmoid focal loss (Lin et al., 2017) for multi-label. mean reduction."""

    def __init__(self, gamma: float = 2.0, alpha: Optional[float] = None):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce * (1 - p_t).pow(self.gamma)
        if self.alpha is not None:
            loss = loss * (self.alpha * targets + (1 - self.alpha) * (1 - targets))
        return loss.mean()


class AsymmetricLossWithLogits(nn.Module):
    """ASL (Ridnik et al., ICCV 2021): gamma_neg=4, gamma_pos=0, clip=0.05. mean reduction."""

    def __init__(self, gamma_pos: float = 0.0, gamma_neg: float = 4.0, clip: float = 0.05, eps: float = 1e-8):
        super().__init__()
        self.gamma_pos, self.gamma_neg, self.clip, self.eps = gamma_pos, gamma_neg, clip, eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        p = torch.sigmoid(logits)
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p       # probability shifting (easy negatives -> 0 loss)
        loss_pos = targets * (1 - p).pow(self.gamma_pos) * F.logsigmoid(logits)
        loss_neg = (1 - targets) * p_neg.pow(self.gamma_neg) * torch.log((1 - p_neg).clamp(min=self.eps))
        return -(loss_pos + loss_neg).mean()


def build_loss(name: str) -> nn.Module:
    return {'bce': nn.BCEWithLogitsLoss(), 'focal': FocalLossWithLogits(), 'asl': AsymmetricLossWithLogits()}[name]


def param_groups(model: nn.Module, weight_decay: float) -> List[dict]:
    """Weight decay sirf conv/linear weights pe; biases aur norm (BN/IN) params pe nahi (standard practice)."""
    decay, no_decay = [], []
    for _, p in model.named_parameters():
        if p.requires_grad:
            (no_decay if p.ndim <= 1 else decay).append(p)
    return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]


def autocast_ctx(enabled: bool):
    """torch.cuda.amp.autocast ka modern equivalent (torch.amp)."""
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=enabled)


def make_grad_scaler(enabled: bool):
    try:                                                  # PyTorch >= 2.3
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):                   # purana PyTorch: torch.cuda.amp
        return torch.cuda.amp.GradScaler(enabled=enabled)


@torch.no_grad()
def predict_proba(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Sigmoid probabilities [N, C] (eval mode: dropout off, BN running stats)."""
    model.eval()
    out = []
    for xb, _ in loader:
        with autocast_ctx(USE_AMP):
            logits = model(xb.to(DEVICE, non_blocking=True))
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, N_CLASSES), dtype=np.float32)


def macro_auc(y: np.ndarray, p: np.ndarray) -> float:
    """Macro ROC-AUC sirf un classes pe jinke positives AUR negatives dono hon."""
    aucs = [roc_auc_score(y[:, c], p[:, c]) for c in range(y.shape[1]) if 0 < y[:, c].sum() < len(y)]
    return float(np.mean(aucs)) if aucs else float('nan')


def safe_torch_load(path: Path) -> dict:
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:                                     # bahut purana torch (weights_only arg nahi)
        return torch.load(path, map_location=DEVICE)


def run_path(name: str, seed: int) -> Path:
    return RUN_DIR / f'{name}__seed{seed}'


def load_trained(name: str, seed: int) -> nn.Module:
    ckpt = safe_torch_load(run_path(name, seed) / 'best.pt')
    model = build_model(name, N_CLASSES)
    model.load_state_dict(ckpt['model_state'])
    return model.to(DEVICE).eval()


# ---- loss sanity check: perfect vs random logits ----
_y = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
for _ln in ('bce', 'focal', 'asl'):
    _good, _bad = build_loss(_ln)(torch.tensor([[6.0, -6.0], [-6.0, 6.0]]), _y), build_loss(_ln)(torch.zeros(2, 2), _y)
    assert _good < _bad, _ln
    print(f'{_ln:5s}: loss(good logits) = {_good:.4f} < loss(zero logits) = {_bad:.4f}  ✅')
print(f'Selected loss: {CFG.loss_name}')

In [ ]:
# =====================================================================
# Cell 5.2: Training loop (AMP + grad-clip + OneCycleLR + best-val checkpoint + early stopping)
# =====================================================================
def train_model(name: str, seed: int, cfg: Config = CFG) -> dict:
    out_dir = run_path(name, seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_file, hist_file = out_dir / 'best.pt', out_dir / 'history.json'
    if ckpt_file.exists() and hist_file.exists() and not cfg.force_retrain:
        try:                                                          # checkpoint current architecture/classes se match?
            build_model(name, N_CLASSES, cfg).load_state_dict(safe_torch_load(ckpt_file)['model_state'])
            print(f'[skip] {name} seed={seed}: checkpoint already hai -> {ckpt_file}')
            return json.loads(hist_file.read_text())
        except (RuntimeError, KeyError) as e:
            print(f'⚠️ {name} seed={seed}: purana checkpoint current model se match nahi karta -> retrain ({type(e).__name__})')

    seed_everything(seed, cfg.deterministic)
    model = build_model(name, N_CLASSES, cfg).to(DEVICE)
    train_loader = make_loader(X_ptb, Y_ptb, idx_train, cfg.batch_size, shuffle=True, seed=seed, drop_last=True)
    val_loader = make_loader(X_ptb, Y_ptb, idx_val, cfg.eval_batch_size, shuffle=False)
    y_val = Y_ptb[idx_val]
    criterion = build_loss(cfg.loss_name)
    optimizer = torch.optim.AdamW(param_groups(model, cfg.weight_decay), lr=cfg.max_lr)
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg.max_lr, pct_start=cfg.pct_start,
                                                    total_steps=cfg.epochs * steps_per_epoch)
    scaler = make_grad_scaler(USE_AMP)

    best, best_epoch, bad, history = -np.inf, 0, 0, []
    t_start = time.time()
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0, loss_sum, n_seen = time.time(), 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)                      # [B, 12, 1000]
            yb = yb.to(DEVICE, non_blocking=True)                      # [B, C]
            optimizer.zero_grad(set_to_none=True)
            with autocast_ctx(USE_AMP):
                logits = model(xb)                                     # [B, C]
            loss = criterion(logits.float(), yb)                       # loss hamesha FP32
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)                                 # clip se pehle unscale zaroori
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()                                           # OneCycle: per-batch step
            loss_sum += loss.item() * xb.size(0)
            n_seen += xb.size(0)

        val_auc = macro_auc(y_val, predict_proba(model, val_loader))
        score = val_auc if np.isfinite(val_auc) else -np.inf
        rec = {'epoch': epoch, 'train_loss': loss_sum / max(n_seen, 1), 'val_macro_auc': val_auc,
               'lr': scheduler.get_last_lr()[0], 'sec': time.time() - t0}
        history.append(rec)
        improved = score > best + 1e-4 or best_epoch == 0
        if improved:
            best, best_epoch, bad = score, epoch, 0
            torch.save({'model_state': model.state_dict(),
                        'meta': {'model': name, 'seed': seed, 'epoch': epoch, 'val_macro_auc': float(val_auc),
                                 'classes': list(CLASSES), 'label_set': cfg.label_set}}, ckpt_file)
        else:
            bad += 1
        print(f'  {name:18s} s{seed} | ep {epoch:02d}/{cfg.epochs} | train loss {rec["train_loss"]:.4f} | '
              f'val macro-AUC {val_auc:.4f} {"★" if improved else " "} | lr {rec["lr"]:.2e} | {rec["sec"]:.0f}s')
        if bad >= cfg.patience:
            print(f'  ⏹ early stopping: {cfg.patience} epochs se improvement nahi; best epoch = {best_epoch}')
            break

    summary = {'model': name, 'seed': seed, 'best_val_macro_auc': float(best), 'best_epoch': best_epoch,
               'epochs_run': len(history), 'train_minutes': (time.time() - t_start) / 60,
               'params': count_params(model), 'history': history}
    hist_file.write_text(json.dumps(summary, indent=1))
    del model, optimizer
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return summary


print('Parity contract (saare models):')
print(f'  loss={CFG.loss_name} | AdamW(wd={CFG.weight_decay}) | OneCycleLR(max_lr={CFG.max_lr}, pct_start={CFG.pct_start}) | '
      f'clip={CFG.grad_clip} | AMP={USE_AMP} | batch={CFG.batch_size} | epochs<= {CFG.epochs} | patience={CFG.patience}')
print(f'  train={len(idx_train):,} (folds 1-8) | val={len(idx_val):,} (fold 9) | steps/epoch={len(idx_train) // CFG.batch_size}')

In [ ]:
# =====================================================================
# Cell 5.3: Train all models x seeds (+ learning curves)
# =====================================================================
TRAIN_SUMMARY: List[dict] = []
for name in CFG.models:
    for seed in CFG.seeds:
        print(f'\n=== Training {name} (seed {seed}) ===')
        TRAIN_SUMMARY.append(train_model(name, seed))

train_table = pd.DataFrame([{k: s[k] for k in ('model', 'seed', 'params', 'best_val_macro_auc', 'best_epoch',
                                                'epochs_run', 'train_minutes')} for s in TRAIN_SUMMARY])
show(train_table.set_index(['model', 'seed']))
train_table.to_csv(TAB_DIR / 'training_summary.csv', index=False)

# ---- Learning curves (first seed); two separate panels, never a dual axis ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for s in TRAIN_SUMMARY:
    if s['seed'] != CFG.seeds[0]:
        continue
    h = pd.DataFrame(s['history'])
    col = MODEL_COLORS.get(s['model'], INK2)
    axes[0].plot(h.epoch, h.train_loss, color=col, label=s['model'])
    axes[1].plot(h.epoch, h.val_macro_auc, color=col, label=s['model'])
    axes[1].scatter([s['best_epoch']], [h.val_macro_auc.iloc[s['best_epoch'] - 1]], s=64, color=col,
                    edgecolor='white', linewidth=2, zorder=3)
axes[0].set_title('Training loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel(f'{CFG.loss_name} loss')
axes[1].set_title('Validation (fold 9) macro-AUC  (● = selected checkpoint)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro-AUC')
_h, _l = axes[1].get_legend_handles_labels()
fig.legend(_h, _l, loc='lower center', ncol=len(_l), bbox_to_anchor=(0.5, -0.06))
fig.tight_layout()
savefig(fig, 'fig05_learning_curves')

---
# Cell 6 — Benchmarking & Cross-Dataset Evaluation 📊

## 6.1 Strict evaluation protocol

1. Har model ke best checkpoint se fold 9 (val), fold 10 (test) aur Chapman (ext) ki probabilities nikaalo.
2. **Thresholds sirf fold 9 pe tune karo:** har class ke liye F1-maximizing threshold. Macro-F1 per-class F1 ka average hai, isliye har class ka F1 alag maximize karne se macro-F1 bhi maximize hota hai.
3. Wahi *exact* thresholds fold 10 aur Chapman pe apply karo. Chapman pe koi tuning nahi hoti (zero-shot).
4. Metrics ke saath **bootstrap 95% CI** aur **paired bootstrap significance test** bhi report karo.

## 6.2 Metrics cheat-sheet

| Metric | Formula / idea | Threshold chahiye? | Prevalence se affect? |
|---|---|---|---|
| **Macro-AUC** (primary) | har class ka ROC-AUC, phir average | ❌ | ❌ (mostly invariant) |
| **Macro-F1** | `2·TP / (2·TP + FP + FN)`, class-wise average | ✅ | ✅ |
| **Sensitivity (Recall)** | `TP / (TP + FN)`: bimaar ko pakadna | ✅ | ❌ |
| **Specificity** | `TN / (TN + FP)`: healthy ko healthy kehna | ✅ | ❌ |
| **AUPRC** | Precision-Recall curve ke neeche ka area | ❌ | ✅ (random baseline = prevalence) |
| **ΔAUC** | `AUC(PTB-XL fold 10) − AUC(Chapman)`: generalization gap | ❌ | ❌ |

> 💡 **Interpretation tip:** Chapman pe F1 girne ki *do* wajah ho sakti hain: (a) model ki ranking kharab ho gayi (tab AUC bhi girega), ya (b) sirf **threshold/calibration shift** hua (AUC same, F1 gira). Isliye hum ek **oracle F1** bhi dikhate hain, jisme thresholds Chapman pe hi tune hote hain. Ye *sirf analysis* ke liye hai, deployable nahi. Oracle F1 − deployable F1 = threshold-transfer loss.

> ⚠️ AUPRC ki random baseline class prevalence hoti hai, aur prevalence PTB-XL aur Chapman mein bahut alag hai. Isliye AUPRC ko datasets ke beech directly compare mat karna; models ko *same dataset* pe compare karo.

### 📖 Glossary: Cell 6

| Term | Hinglish explanation |
|---|---|
| **ROC curve** | Har threshold pe (False Positive Rate, True Positive Rate) ka plot. |
| **AUC / AUROC** | ROC curve ke neeche ka area = probability ki ek random positive record ko model ek random negative se upar rank karega. 0.5 = random, 1.0 = perfect. |
| **Macro average** | Har class ka metric alag nikaalo, phir simple average lo (rare classes ko bhi barabar weight milta hai). |
| **TP / FP / FN / TN** | True Positive, False Positive, False Negative, True Negative. |
| **Precision** | `TP / (TP + FP)`: jitno ko "positive" bola, unme se kitne sach mein positive the. |
| **F1 score** | Precision aur Recall ka harmonic mean. |
| **AUPRC / Average Precision** | PR-curve ke neeche ka area; rare classes ke liye AUC se zyada strict. |
| **Threshold tuning** | Probability ko 0/1 mein badalne ka cut-off choose karna (0.5 hamesha best nahi hota). |
| **Bootstrap 95% CI** | Test set ko replacement ke saath 1000 baar resample karke metric ka spread dekhna: "result kitna stable hai". |
| **Paired bootstrap test** | Dono models ko *same* resamples pe compare karna; p-value = kitni baar difference ka sign ulta aaya. |
| **p-value** | Agar asal mein koi farak na ho, to itna (ya isse zyada) farak sirf chance se dikhne ki probability. p < 0.05 ⇒ "statistically significant". |
| **Generalization gap (ΔAUC)** | In-domain aur external performance ka farak. Chhota gap = better generalization. |
| **Calibration** | Predicted probability aur actual frequency ka match (model 0.8 bole to ~80% cases sach mein positive hon). |
| **Oracle threshold** | Test set pe hi tune kiya gaya threshold. Real deployment mein possible nahi, sirf analysis ke liye. |

In [ ]:
# =====================================================================
# Cell 6.1: Predictions for val / test / Chapman (cached per model & seed)
# =====================================================================
def collect_predictions(name: str, seed: int) -> Dict[str, np.ndarray]:
    ckpt = run_path(name, seed) / 'best.pt'
    pred_file = run_path(name, seed) / 'preds.npz'
    if pred_file.exists() and pred_file.stat().st_mtime >= ckpt.stat().st_mtime and not CFG.force_retrain:
        return dict(np.load(pred_file))
    model = load_trained(name, seed)
    P = {}
    for split, X_, Y_, idx_ in [('val', X_ptb, Y_ptb, idx_val), ('test', X_ptb, Y_ptb, idx_test),
                                ('ext', X_chap, Y_chap, idx_ext)]:
        t0 = time.time()
        P[f'{split}_p'] = predict_proba(model, make_loader(X_, Y_, idx_, CFG.eval_batch_size, shuffle=False))
        P[f'{split}_y'] = Y_[idx_]
        P[f'{split}_ms_per_ecg'] = np.array(1000 * (time.time() - t0) / max(len(idx_), 1))
    np.savez_compressed(pred_file, **P)
    return P


PREDS: Dict[Tuple[str, int], Dict[str, np.ndarray]] = {}
for name in CFG.models:
    for seed in CFG.seeds:
        PREDS[(name, seed)] = collect_predictions(name, seed)
        P = PREDS[(name, seed)]
        print(f'{name:18s} s{seed}: val {P["val_p"].shape} | test {P["test_p"].shape} | Chapman {P["ext_p"].shape} '
              f'| inference {float(P["ext_ms_per_ecg"]):.2f} ms/ECG')

In [ ]:
# =====================================================================
# Cell 6.2: Threshold tuning, metrics, fast paired bootstrap
# =====================================================================
def tune_thresholds(y: np.ndarray, p: np.ndarray) -> np.ndarray:
    """Per-class F1-optimal threshold (exact, via precision_recall_curve). Use ONLY on fold 9."""
    thr = np.full(y.shape[1], 0.5)
    for c in range(y.shape[1]):
        if 0 < y[:, c].sum() < len(y):
            prec, rec, t = precision_recall_curve(y[:, c], p[:, c])
            f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
            thr[c] = t[np.argmax(f1[:-1])]                    # last PR point (rec=0) ka threshold nahi hota
    return thr


def per_class_metrics(y: np.ndarray, p: np.ndarray, thr: np.ndarray) -> pd.DataFrame:
    yhat = p >= thr[None, :]
    rows = []
    for c, cls in enumerate(CLASSES):
        yt, yp, s = y[:, c].astype(bool), yhat[:, c], p[:, c]
        tp, fp = int((yt & yp).sum()), int((~yt & yp).sum())
        fn, tn = int((yt & ~yp).sum()), int((~yt & ~yp).sum())
        valid = 0 < yt.sum() < len(yt)
        rows.append({'class': cls, 'n_pos': int(yt.sum()), 'thr': float(thr[c]),
                     'AUC': roc_auc_score(yt, s) if valid else np.nan,
                     'AUPRC': average_precision_score(yt, s) if valid else np.nan,
                     'F1': 2 * tp / (2 * tp + fp + fn) if valid else np.nan,
                     'Sens': tp / (tp + fn) if valid else np.nan,
                     'Spec': tn / (tn + fp) if valid else np.nan})
    return pd.DataFrame(rows).set_index('class')


METRICS = ['AUC', 'F1', 'Sens', 'Spec', 'AUPRC']


def macro_of(df: pd.DataFrame) -> Dict[str, float]:
    return {m: float(df[m].mean(skipna=True)) for m in METRICS}


def _weighted_auc(y_c: np.ndarray, s_c: np.ndarray, W: np.ndarray, order: np.ndarray) -> np.ndarray:
    """Tie-aware AUC for many bootstrap replicates at once. W: [B, n] resample counts -> [B] AUCs."""
    s, yy = s_c[order], y_c[order].astype(bool)
    starts = np.r_[0, np.flatnonzero(np.diff(s)) + 1]                        # equal-score groups (ties)
    Wo = W[:, order]
    wp = np.add.reduceat(Wo * yy, starts, axis=1)                             # positive weight per group
    wn = np.add.reduceat(Wo * ~yy, starts, axis=1)                            # negative weight per group
    below = np.cumsum(wn, axis=1) - wn                                        # negatives strictly lower score
    num = (wp * (below + 0.5 * wn)).sum(1)
    den = wp.sum(1) * wn.sum(1)
    return np.where(den > 0, num / np.maximum(den, 1e-12), np.nan)


def bootstrap_auc(y: np.ndarray, p_list: List[np.ndarray], n_boot: int, seed: int, chunk: int = 100) -> List[np.ndarray]:
    """PAIRED bootstrap: har prediction set ke liye SAME resamples. Returns list of [n_boot, C] AUC matrices."""
    n, C = y.shape
    rng = np.random.default_rng(seed)
    orders = [[np.argsort(p[:, c], kind='mergesort') for c in range(C)] for p in p_list]
    out = [np.empty((n_boot, C)) for _ in p_list]
    for b0 in range(0, n_boot, chunk):
        nb = min(chunk, n_boot - b0)
        W = np.stack([np.bincount(rng.integers(0, n, n), minlength=n) for _ in range(nb)]).astype(np.float32)
        for k, p in enumerate(p_list):
            for c in range(C):
                out[k][b0:b0 + nb, c] = _weighted_auc(y[:, c], p[:, c], W, orders[k][c])
    return out


def macro_rows(M: np.ndarray) -> np.ndarray:
    """Bootstrap rows jahan kisi class ke positives/negatives na hon, drop (Strodthoff et al. 2021 style)."""
    return M[~np.isnan(M).any(1)].mean(1)


def ci95(v: np.ndarray) -> Tuple[float, float]:
    v = np.asarray(v)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)


# ---- sanity: fast bootstrap AUC (weights = 1) == sklearn ----
_rng = np.random.default_rng(0)
_yc = (_rng.random(500) < 0.2).astype(float)
_sc = np.round(_rng.random(500) + 0.5 * _yc, 2)                               # rounding => ties bhi test
_fast = _weighted_auc(_yc, _sc, np.ones((1, 500), np.float32), np.argsort(_sc, kind='mergesort'))[0]
assert abs(_fast - roc_auc_score(_yc, _sc)) < 1e-9, (_fast, roc_auc_score(_yc, _sc))
print(f'✅ fast tie-aware bootstrap AUC matches sklearn ({_fast:.6f})')

In [ ]:
# =====================================================================
# Cell 6.3: Evaluate every model (fold-9 thresholds -> fold 10 + Chapman) + bootstrap
# =====================================================================
RESULTS: Dict[str, List[dict]] = {}
THRESHOLDS: Dict[str, Dict[str, float]] = {}
for name in CFG.models:
    RESULTS[name] = []
    for seed in CFG.seeds:
        P = PREDS[(name, seed)]
        thr = tune_thresholds(P['val_y'], P['val_p'])                        # ONLY fold 9
        RESULTS[name].append({
            'thr': thr,
            'test': per_class_metrics(P['test_y'], P['test_p'], thr),        # fold 10, same thresholds
            'ext': per_class_metrics(P['ext_y'], P['ext_p'], thr),           # Chapman, same thresholds (zero-shot)
            'ext_oracle': per_class_metrics(P['ext_y'], P['ext_p'], tune_thresholds(P['ext_y'], P['ext_p'])),
        })
        THRESHOLDS[f'{name}__seed{seed}'] = dict(zip(CLASSES, map(float, thr)))
(TAB_DIR / 'thresholds_fold9.json').write_text(json.dumps(THRESHOLDS, indent=1))

_keys = [(n, s) for n in CFG.models for s in CFG.seeds]
t0 = time.time()
_bt = bootstrap_auc(Y_ptb[idx_test], [PREDS[k]['test_p'] for k in _keys], CFG.n_boot, CFG.boot_seed)
_be = bootstrap_auc(Y_chap[idx_ext], [PREDS[k]['ext_p'] for k in _keys], CFG.n_boot, CFG.boot_seed + 1)
BOOT = {n: {'test': np.mean([_bt[_keys.index((n, s))] for s in CFG.seeds], 0),        # seed-averaged per replicate
            'ext': np.mean([_be[_keys.index((n, s))] for s in CFG.seeds], 0)} for n in CFG.models}
print(f'Bootstrap ({CFG.n_boot} paired resamples x {len(_keys)} runs x 2 test sets): {time.time() - t0:.1f}s')

In [ ]:
# =====================================================================
# Cell 6.4: MAIN BENCHMARK TABLE + paired significance tests
# =====================================================================
def mean_sd(vals: List[float]) -> str:
    vals = np.asarray(vals, dtype=float)
    return f'{vals.mean():.4f}' if len(vals) == 1 else f'{vals.mean():.4f} ± {vals.std():.4f}'


rows = []
for name in CFG.models:
    rs = RESULTS[name]
    t_macro = [macro_of(r['test']) for r in rs]
    e_macro = [macro_of(r['ext']) for r in rs]
    bt, be = macro_rows(BOOT[name]['test']), macro_rows(BOOT[name]['ext'])
    m = min(len(bt), len(be))
    d_lo, d_hi = ci95(bt[:m] - be[:m])
    t_lo, t_hi = ci95(bt)
    e_lo, e_hi = ci95(be)
    auc_t, auc_e = np.mean([d['AUC'] for d in t_macro]), np.mean([d['AUC'] for d in e_macro])
    row = {'Model': name, '|Params (M)': round(count_params(build_model(name, N_CLASSES)) / 1e6, 3)}
    for tag, mac, lo, hi in [('PTB-XL fold 10 (in-domain)', t_macro, t_lo, t_hi), ('Chapman (zero-shot)', e_macro, e_lo, e_hi)]:
        row[f'{tag}|Macro-AUC'] = mean_sd([d['AUC'] for d in mac])
        row[f'{tag}|AUC 95% CI'] = f'[{lo:.3f}, {hi:.3f}]'
        for met in ('F1', 'Sens', 'Spec', 'AUPRC'):
            row[f'{tag}|Macro-{met}'] = mean_sd([d[met] for d in mac])
    row['Generalization|ΔAUC'] = f'{auc_t - auc_e:+.4f}'
    row['Generalization|ΔAUC 95% CI'] = f'[{d_lo:+.3f}, {d_hi:+.3f}]'
    row['Generalization|Chapman F1 (oracle thr)*'] = mean_sd([macro_of(r['ext_oracle'])['F1'] for r in rs])
    rows.append(row)
BENCH = pd.DataFrame(rows).set_index('Model')
BENCH.columns = pd.MultiIndex.from_tuples([tuple(col.split('|')) for col in BENCH.columns])
print('=' * 120)
print(f'MAIN BENCHMARK | label set: {CFG.label_set} ({N_CLASSES} classes) | thresholds tuned on PTB-XL fold 9 ONLY | '
      f'seeds: {list(CFG.seeds)}')
print('=' * 120)
show(BENCH)
print('* oracle = thresholds tuned on Chapman itself: ANALYSIS ONLY (not deployable). Gap vs Macro-F1 = threshold-transfer loss.')
BENCH.to_csv(TAB_DIR / 'benchmark_summary.csv')
try:
    (TAB_DIR / 'benchmark_summary.tex').write_text(BENCH.to_latex())
except Exception as e:                                                       # jinja2 na ho to LaTeX skip
    print(f'(LaTeX export skipped: {type(e).__name__})')

# ---------- Paired bootstrap significance: proposed vs every other model ----------
REF = 'PhysioDKS-IncepSE'
sig_rows = []
if REF in CFG.models:
    for other in [m for m in CFG.models if m != REF]:
        for split, label in [('test', 'PTB-XL fold 10'), ('ext', 'Chapman')]:
            a, b = BOOT[REF][split], BOOT[other][split]
            ok = ~(np.isnan(a).any(1) | np.isnan(b).any(1))
            diff = a[ok].mean(1) - b[ok].mean(1)
            p = min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()))
            lo, hi = ci95(diff)
            sig_rows.append({'comparison': f'{REF} − {other}', 'test set': label,
                             'Δ Macro-AUC': f'{diff.mean():+.4f}', '95% CI': f'[{lo:+.4f}, {hi:+.4f}]',
                             'p (two-sided)': f'< {1 / ok.sum():.3f}' if p == 0 else f'{p:.4f}'})
        # generalization gap comparison: kya proposed ka ΔAUC chhota hai?
        ga = BOOT[REF]['test'] - BOOT[REF]['ext']
        gb = BOOT[other]['test'] - BOOT[other]['ext']
        ok = ~(np.isnan(ga).any(1) | np.isnan(gb).any(1))
        dg = ga[ok].mean(1) - gb[ok].mean(1)
        p = min(1.0, 2 * min((dg <= 0).mean(), (dg >= 0).mean()))
        lo, hi = ci95(dg)
        sig_rows.append({'comparison': f'{REF} − {other}', 'test set': 'ΔAUC gap (neg = proposed generalizes better)',
                         'Δ Macro-AUC': f'{dg.mean():+.4f}', '95% CI': f'[{lo:+.4f}, {hi:+.4f}]',
                         'p (two-sided)': f'< {1 / ok.sum():.3f}' if p == 0 else f'{p:.4f}'})
    SIG = pd.DataFrame(sig_rows).set_index(['comparison', 'test set'])
    show(SIG)
    SIG.to_csv(TAB_DIR / 'significance_paired_bootstrap.csv')

# ---------- Per-class AUC table (seed-mean) ----------
PER_CLASS_AUC = {split: pd.DataFrame({n: np.mean([r[split]['AUC'].values for r in RESULTS[n]], 0) for n in CFG.models},
                                     index=CLASSES).T for split in ('test', 'ext')}
PER_CLASS_AUC['delta'] = PER_CLASS_AUC['test'] - PER_CLASS_AUC['ext']
for split, title in [('test', 'Per-class AUC: PTB-XL fold 10'), ('ext', 'Per-class AUC: Chapman (zero-shot)'),
                     ('delta', 'Per-class ΔAUC (fold 10 − Chapman)')]:
    print(f'\n{title}')
    show(PER_CLASS_AUC[split].round(4))
    PER_CLASS_AUC[split].to_csv(TAB_DIR / f'per_class_auc_{split}.csv')

In [ ]:
# =====================================================================
# Cell 6.5: Benchmark figures
# =====================================================================
models = list(CFG.models)

# ---- (1) Dumbbell: in-domain vs zero-shot macro-AUC (with 95% CI) per model ----
fig, ax = plt.subplots(figsize=(9, 0.62 * len(models) + 1.6))
for i, name in enumerate(models):
    vt = np.mean([macro_of(r['test'])['AUC'] for r in RESULTS[name]])
    ve = np.mean([macro_of(r['ext'])['AUC'] for r in RESULTS[name]])
    ax.hlines(i, min(vt, ve), max(vt, ve), color=AXIS, lw=2, zorder=1)
    for v, split, ds in [(vt, 'test', 'PTB-XL'), (ve, 'ext', 'Chapman')]:
        lo, hi = ci95(macro_rows(BOOT[name][split]))
        ax.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt='none', ecolor=DATASET_COLORS[ds], elinewidth=1, capsize=3, zorder=2)
        ax.scatter(v, i, s=72, color=DATASET_COLORS[ds], edgecolor='white', linewidth=2, zorder=3,
                   label=('PTB-XL fold 10 (in-domain)' if ds == 'PTB-XL' else 'Chapman (zero-shot)') if i == 0 else None)
    ax.text(1.02, i, f'ΔAUC {round(vt - ve, 3) + 0.0:+.3f}', transform=ax.get_yaxis_transform(), va='center',
            fontsize=9, color=INK2)
ax.set_yticks(range(len(models))); ax.set_yticklabels(models); ax.invert_yaxis()
ax.set_xlabel('Macro-AUC (point = estimate, whisker = bootstrap 95% CI)')
ax.set_title('In-domain vs zero-shot cross-dataset performance')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
savefig(fig, 'fig06a_benchmark_dumbbell')

# ---- (2) Per-class AUC heatmaps + ΔAUC (diverging) ----
fig, axes = plt.subplots(1, 3, figsize=(18, 0.5 * len(models) + 2.4))
for ax, key, title, cmap, vlim in [
        (axes[0], 'test', 'AUC: PTB-XL fold 10', 'Blues', (0.5, 1.0)),
        (axes[1], 'ext', 'AUC: Chapman (zero-shot)', 'Blues', (0.5, 1.0)),
        (axes[2], 'delta', 'ΔAUC = fold 10 − Chapman', 'RdBu_r', None)]:
    Mx = PER_CLASS_AUC[key].values
    if vlim is None:
        r = np.nanmax(np.abs(Mx)) if np.isfinite(Mx).any() else 0.1
        vlim = (-r, r)
    im = ax.imshow(Mx, aspect='auto', cmap=cmap, vmin=vlim[0], vmax=vlim[1])
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(len(models))); ax.set_yticklabels(models if ax is axes[0] else [])
    ax.grid(False); ax.set_title(title)
    for i in range(Mx.shape[0]):
        for j in range(Mx.shape[1]):
            if np.isfinite(Mx[i, j]):
                frac = (Mx[i, j] - vlim[0]) / (vlim[1] - vlim[0] + 1e-12)
                dark = frac > 0.65 or (key == 'delta' and abs(frac - 0.5) > 0.35)
                ax.text(j, i, f'{Mx[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white' if dark else INK)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
savefig(fig, 'fig06b_per_class_auc')

# ---- (3) ROC curves on Chapman: small multiples (one panel per class) ----
ncol = 3
nrow = int(np.ceil(N_CLASSES / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow), squeeze=False)
for c, cls in enumerate(CLASSES):
    ax = axes[c // ncol, c % ncol]
    y_c = Y_chap[idx_ext][:, c]
    ax.plot([0, 1], [0, 1], color=AXIS, lw=0.8)
    if 0 < y_c.sum() < len(y_c):
        for name in models:
            fpr, tpr, _ = roc_curve(y_c, PREDS[(name, CFG.seeds[0])]['ext_p'][:, c])
            ax.plot(fpr, tpr, color=MODEL_COLORS.get(name, INK2), lw=1.4, label=name)
    ax.set_title(f'{cls} (n+ = {int(y_c.sum())})')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
for k in range(N_CLASSES, nrow * ncol):
    axes[k // ncol, k % ncol].axis('off')
handles, labels = next((a.get_legend_handles_labels() for a in axes.flat if a.get_legend_handles_labels()[0]), ([], []))
if handles:
    fig.legend(handles, labels, loc='lower center', ncol=len(models), bbox_to_anchor=(0.5, -0.02))
fig.suptitle('ROC curves on Chapman (zero-shot, seed %d)' % CFG.seeds[0], y=1.01)
fig.tight_layout()
savefig(fig, 'fig06c_roc_chapman')

# ---- (4) Threshold transfer: deployable (fold-9 thr) vs oracle F1 on Chapman, proposed model ----
_m = REF if REF in CFG.models else models[-1]
f1_dep = np.mean([r['ext']['F1'].values for r in RESULTS[_m]], 0)
f1_orc = np.mean([r['ext_oracle']['F1'].values for r in RESULTS[_m]], 0)
fig, ax = plt.subplots(figsize=(9, 3.8))
xx = np.arange(N_CLASSES)
ax.vlines(xx, np.minimum(f1_dep, f1_orc), np.maximum(f1_dep, f1_orc), color=AXIS, lw=2)
ax.scatter(xx, f1_orc, s=64, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='Oracle threshold (analysis only)')
ax.scatter(xx, f1_dep, s=64, color=MODEL_COLORS.get(_m, INK), edgecolor='white', linewidth=2, zorder=3,
           label='Fold-9 threshold (deployable)')
ax.set_xticks(xx); ax.set_xticklabels(CLASSES); ax.set_ylabel('F1 on Chapman'); ax.set_ylim(0, 1.02)
ax.set_title(f'Threshold-transfer loss: {_m}')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
savefig(fig, 'fig06d_threshold_transfer')

---
# Cell 7 — Explainable AI: 1D Grad-CAM, SE Lead-Attention, DKS Attention 🔍

## 7.1 1D Grad-CAM: kaise kaam karta hai

Target layer ke activations `A_k(t)` (channel *k*, time *t*) aur class score `y_c` ke gradient se:
- **Channel weight:** `α_k = mean_t ∂y_c / ∂A_k(t)`, yaani channel *k* class *c* ke liye kitna important hai
- **CAM:** `CAM(t) = ReLU( Σ_k α_k · A_k(t) )`, phir 1000 samples tak upsample aur [0, 1] mein normalize

Hum CAM teen jagah se nikaalte hain:

| Model / layer | Resolution | Lead-resolved? |
|---|---|---|
| Baseline ResNet1D: last residual stage | 125 steps (80 ms), upsampled | ❌ (ek heatmap, saari leads pe same) |
| PhysioDKS-IncepSE: final IncepSE stage | 1000 steps (10 ms) | ❌ |
| PhysioDKS-IncepSE: **Stage-1 output** | 1000 steps | ✅ **har lead ka alag heatmap** (lead-grouped channels ki wajah se) |

> ⚠️ **Fair-comparison caveat:** ResNet ka CAM coarse (125 steps) hai, isliye upsample hone pe "blurry" dikhega. Ye resolution ka effect hai, model ki "galti" nahi. Isi wajah se visual ke saath **quantitative** metrics bhi report karte hain.

## 7.2 Example selection rule (no cherry-picking)

Har XAI class ke liye Chapman ke positive records mein se (preferably jinme sirf wahi ek class positive ho) **fixed seed** se random pick hota hai. Rule results dekhne se pehle fix hai; examples results dekh kar nahi chune gaye.

## 7.3 Quantitative saliency metrics (visual "proof" ko numbers mein badalna)

R-peaks detect karke (multi-lead slope-energy envelope) do regions banate hain:
- **QRS window:** R ± 60 ms
- **PQRST window:** R − 250 ms ... R + 450 ms (P-wave se T-wave tak)

| Metric | Definition | Interpretation |
|---|---|---|
| **QRS enrichment** | (CAM mass in QRS) / (QRS ka time-fraction) | > 1 ⇒ QRS pe chance se zyada focus |
| **PQRST enrichment** | (CAM mass in PQRST) / (PQRST ka time-fraction) | > 1 ⇒ waveforms pe focus, baseline pe kam |
| **Off-wave fraction** | 1 − (CAM mass in PQRST) | baseline/isoelectric segments pe kitna attention gaya |

Baseline vs proposed ko same records pe compare karne ke liye paired **Wilcoxon signed-rank test** use hota hai. Table mein `NaN` ka matlab: us class ke records mein CAM poora khaali tha (ReLU ke baad zero, yaani model ko class ke liye *positive* evidence nahi mila) ya < 2 beats detect hue.

> 🩺 **Clinical nuance (paper ke liye important):** "Baseline pe attention = bura" har class ke liye sach nahi hai! **AFIB/AFL** mein diagnostic information *TP-segment / baseline* mein hi hoti hai (fibrillatory f-waves, flutter F-waves), aur RR irregularity mein. Isliye XAI claims **class-conditional** likho: BBB/PVC ke liye QRS focus expected hai; AFIB/AFL ke liye off-wave attention clinically valid ho sakta hai. Tachycardia mein PQRST windows lagbhag poora signal cover kar lete hain, isliye wahan enrichment ≈ 1 aana normal hai.

## 7.4 Lead importance: do complementary views

| View | Source | Class-specific? |
|---|---|---|
| **Lead-SE gates** | Stage-1 ka 12-lead sigmoid gate | ❌ input-dependent, lekin class-agnostic |
| **Lead-resolved Grad-CAM share** | Stage-1 CAM ka per-lead mass | ✅ |

**DKS attention ko teen tarah se dekhte hain:** (1) class-wise average (saare channels ka average; ye *lossy* hai, kyunki alag channels opposite directions mein shift ho kar cancel ho sakte hain); (2) **per-lead branch-preference map** = (k=5 weight − k=21 weight), yaani har lead pe QRS-scale vs P/T-scale filter ki preference; (3) **dynamic range** = har channel ka branch-weight records ke beech kitna badla (max − min). Agar dynamic range ≈ 0 ho, to DKS ek static band-weighting ban gaya. Ye bhi ek valid finding hai, aur `PhysioDKS-noDKS` ablation batata hai ki performance pe iska kitna asar hai.

Textbook expectations (compare karne ke liye):

| Class | Expected key leads |
|---|---|
| RBBB | V1, V2 (rSR′); I, V6 (wide slurred S) |
| LBBB | I, aVL, V5, V6 (broad notched R); V1 (deep QS/rS) |
| AFIB | II, V1 (f-waves sabse clear) + saari leads (RR irregularity) |
| AFL | II, III, aVF (sawtooth), V1 |
| PAC / PVC | II, V1 (premature beat ki morphology) |
| STach / SBrady / NSR | II (P-wave + rate); rate saari leads mein dikhta hai |

### 📖 Glossary: Cell 7

| Term | Hinglish explanation |
|---|---|
| **Grad-CAM** | Gradient-weighted Class Activation Mapping: gradients se pata lagao ki kaunsa time-region class score badhata hai. |
| **Saliency / attribution map** | Input ke har point ki "importance" ka heatmap. |
| **Activation / feature map** | Kisi layer ka output `[B, C, T]`. |
| **Gradient** | Output kitna badlega agar input/activation thoda badle (derivative). |
| **Forward hook** | PyTorch feature: kisi layer ka output bina model ka code badle "pakad" lena. |
| **Lead-resolved CAM** | Har lead ka alag heatmap (sirf tab possible jab channels lead-grouped hon). |
| **P wave / QRS complex / T wave** | P = atria ka depolarization (~80–110 ms); QRS = ventricles ka depolarization (~80–120 ms); T = ventricles ka repolarization. |
| **R-peak** | QRS ka sabse ooncha point; beats detect karne ke liye use hota hai. |
| **RR interval** | Do R-peaks ke beech ka time (heart rate = 60 / RR). |
| **TP segment / isoelectric baseline** | T-wave khatam hone se agli P-wave tak ka flat hissa. |
| **Enrichment ratio** | Observed / chance. 1 se zyada = us region pe "focus". |
| **Wilcoxon signed-rank test** | Paired, non-parametric test: kya do models ke per-record scores systematically alag hain? |
| **Frequency response** | Filter kaunsi frequencies pass karta hai aur kaunsi attenuate karta hai. |
| **Spectral centroid** | Frequency response ka "center of mass" (Hz): filter roughly kis band pe kaam karta hai. |

In [ ]:
# =====================================================================
# Cell 7.1: XAI toolkit: Grad-CAM, activation capture, R-peaks, saliency metrics
# =====================================================================
class GradCAM1D:
    """1D Grad-CAM (Selvaraju et al., 2017) for time series. target_layer output: [B, K, T'].
    lead_groups=12 => lead-resolved CAM [B, 12, T] (valid only for lead-grouped channels, e.g. PhysioDKS Stage-1)."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model, self._acts = model, None
        self._handle = target_layer.register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        self._acts = output

    def remove(self) -> None:
        self._handle.remove()

    def __call__(self, x: torch.Tensor, class_idx: int, lead_groups: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
        self.model.eval()
        x = x.to(DEVICE, dtype=torch.float32)                         # [B, 12, T]
        with torch.enable_grad():
            logits = self.model(x)                                     # [B, C]
            score = logits[:, class_idx].sum()                         # eval mode => samples independent => sum safe
            grads = torch.autograd.grad(score, self._acts)[0]          # [B, K, T']
        A, G = self._acts.detach().float(), grads.detach().float()
        alpha = G.mean(dim=-1, keepdim=True)                           # [B, K, 1] channel importance
        cam = alpha * A                                                # [B, K, T']
        B, K, Tp = cam.shape
        cam = cam.view(B, lead_groups, K // lead_groups, Tp).sum(2) if lead_groups else cam.sum(1, keepdim=True)
        cam = F.interpolate(F.relu(cam), size=x.shape[-1], mode='linear', align_corners=False)   # [B, L, T]
        cam = cam / cam.flatten(1).amax(1).clamp_min(1e-12).view(B, 1, 1)                        # per-record [0, 1]
        return cam.cpu().numpy(), torch.sigmoid(logits.detach().float()).cpu().numpy()


class ActivationCapture:
    """Context manager: forward hook se kisi module ka output collect karo (Lead-SE gates, DKS softmax)."""

    def __init__(self, module: nn.Module):
        self.module, self.outputs = module, []

    def __enter__(self):
        self._h = self.module.register_forward_hook(lambda m, i, o: self.outputs.append(o.detach().float().cpu()))
        return self

    def __exit__(self, *exc):
        self._h.remove()

    def cat(self) -> torch.Tensor:
        return torch.cat(self.outputs, 0)


def detect_r_peaks(sig12: np.ndarray, fs: int = CFG.fs) -> np.ndarray:
    """Multi-lead slope-energy envelope pe peak detection (Pan-Tompkins-lite). sig12: [12, T] z-scored."""
    d = np.abs(np.diff(sig12, axis=1, prepend=sig12[:, :1])).sum(0)            # [T] (QRS mein slope max)
    env = np.convolve(d, np.ones(5) / 5, mode='same')                           # 50 ms smoothing
    peaks, _ = find_peaks(env, distance=int(0.3 * fs), height=0.35 * np.percentile(env, 99))
    return peaks


def wave_masks(T: int, r_peaks: np.ndarray, fs: int = CFG.fs) -> Tuple[np.ndarray, np.ndarray]:
    qrs, pqrst = np.zeros(T, bool), np.zeros(T, bool)
    for r in r_peaks:
        qrs[max(0, r - int(0.06 * fs)): r + int(0.06 * fs) + 1] = True        # R ± 60 ms
        pqrst[max(0, r - int(0.25 * fs)): r + int(0.45 * fs) + 1] = True      # P (−250 ms) ... T (+450 ms)
    return qrs, pqrst


def saliency_stats(sig12: np.ndarray, cam: np.ndarray) -> Dict[str, float]:
    """cam: [T] >= 0. Enrichment metrics (NaN agar < 2 beats detect hon ya CAM khaali ho)."""
    r = detect_r_peaks(sig12)
    if len(r) < 2 or cam.sum() <= 0:
        return {'qrs_enrich': np.nan, 'pqrst_enrich': np.nan, 'off_wave_frac': np.nan}
    qrs, pqrst = wave_masks(len(cam), r)
    mass = cam / cam.sum()
    return {'qrs_enrich': mass[qrs].sum() / qrs.mean(), 'pqrst_enrich': mass[pqrst].sum() / pqrst.mean(),
            'off_wave_frac': 1.0 - mass[pqrst].sum()}


def batched_cam(cam_obj: GradCAM1D, X_sel: np.ndarray, c: int, lead_groups: Optional[int] = None,
                bs: int = 32) -> Tuple[np.ndarray, np.ndarray]:
    cams, probs = [], []
    for s in range(0, len(X_sel), bs):
        cm, pr = cam_obj(torch.from_numpy(X_sel[s:s + bs]), c, lead_groups)
        cams.append(cm)
        probs.append(pr[:, c])
    return np.concatenate(cams), np.concatenate(probs)


# ---- Models for XAI: baseline vs proposed (first seed) ----
BASELINE, PROPOSED = 'ResNet1D', 'PhysioDKS-IncepSE'
assert BASELINE in CFG.models and PROPOSED in CFG.models, 'XAI ke liye ResNet1D aur PhysioDKS-IncepSE dono chahiye'
base_model = load_trained(BASELINE, CFG.seeds[0])
prop_model = load_trained(PROPOSED, CFG.seeds[0])
cam_base = GradCAM1D(base_model, base_model.gradcam_targets()['final'])
cam_prop = GradCAM1D(prop_model, prop_model.gradcam_targets()['final'])
cam_s1 = GradCAM1D(prop_model, prop_model.gradcam_targets()['stage1'])

# ---- sanity check: Grad-CAM shapes on a dummy batch ----
_cm, _pr = cam_s1(torch.randn(2, CFG.n_leads, CFG.n_samples), 0, lead_groups=CFG.n_leads)
assert _cm.shape == (2, CFG.n_leads, CFG.n_samples) and _pr.shape == (2, N_CLASSES)
_cm, _ = cam_base(torch.randn(2, CFG.n_leads, CFG.n_samples), 0)
assert _cm.shape == (2, 1, CFG.n_samples) and np.nanmax(_cm) <= 1.0 + 1e-6
print(f'✅ Grad-CAM ready | baseline CAM {(2, 1, CFG.n_samples)} | proposed Stage-1 lead-resolved CAM {(2, CFG.n_leads, CFG.n_samples)}')

XAI_DEFAULT = {'rhythm9': ['RBBB', 'LBBB', 'AFIB', 'PVC'], 'superclass5': ['CD', 'MI', 'STTC', 'HYP']}
y_ext = Y_chap[idx_ext]
XAI_CLASSES = [c for c in XAI_DEFAULT[CFG.label_set] if c in CLASSES and y_ext[:, CLASSES.index(c)].sum() > 0]
print('XAI classes (Chapman positives available):', XAI_CLASSES)

In [ ]:
# =====================================================================
# Cell 7.2: Side-by-side 12-lead Grad-CAM: Baseline vs PhysioDKS-IncepSE (same Chapman ECGs)
# =====================================================================
def pick_examples(cls: str, n: int, seed: int = 0) -> np.ndarray:
    """Fixed-seed selection rule (no cherry-picking). Returns positions inside idx_ext."""
    c = CLASSES.index(cls)
    pos = np.where(y_ext[:, c] == 1)[0]
    single = pos[y_ext[pos].sum(1) == 1]                              # sirf yahi class positive (clean example)
    pool = single if len(single) >= n else pos
    return np.sort(np.random.default_rng(seed + 17 * c).choice(pool, size=min(n, len(pool)), replace=False))


def plot_cam_column(axes, sig12: np.ndarray, cam: np.ndarray, title: str, r_peaks: np.ndarray, cmap: str = 'Reds') -> None:
    """axes: 12 Axes (ek column). sig12 [12, T]; cam [1, T] (shared) ya [12, T] (lead-resolved)."""
    T = sig12.shape[-1]
    t = np.arange(T) / CFG.fs
    for i, ax in enumerate(axes):
        y = sig12[i]
        c = cam[0] if cam.shape[0] == 1 else cam[i]
        lo, hi = float(y.min()), float(y.max())
        pad = 0.15 * (hi - lo + 1e-6)
        ax.imshow(c[None, :], aspect='auto', cmap=cmap, vmin=0, vmax=1, extent=[0, T / CFG.fs, lo - pad, hi + pad],
                  interpolation='bilinear', alpha=0.9)
        for r in r_peaks:
            ax.axvline(r / CFG.fs, color=AXIS, lw=0.4, zorder=1)
        ax.plot(t, y, color=INK, lw=0.6, zorder=2)
        ax.set_xlim(0, T / CFG.fs); ax.set_ylim(lo - pad, hi + pad)
        ax.set_yticks([]); ax.grid(False)
        ax.set_ylabel(LEADS[i], rotation=0, ha='right', va='center', fontsize=8, color=INK2)
        if i < len(axes) - 1:
            ax.tick_params(labelbottom=False)
    axes[0].set_title(title, fontsize=9)
    axes[-1].set_xlabel('Time (s)')


for cls in XAI_CLASSES:
    c = CLASSES.index(cls)
    for pos in pick_examples(cls, CFG.xai_n_examples):
        rec_i = idx_ext[pos]
        sig = np.array(X_chap[rec_i], dtype=np.float32)               # [12, 1000]
        x = torch.from_numpy(sig)[None]                               # [1, 12, 1000]
        cb, pb = cam_base(x, c)                                       # [1, 1, 1000]
        cf, pf = cam_prop(x, c)                                       # [1, 1, 1000]
        cs, _ = cam_s1(x, c, lead_groups=CFG.n_leads)                 # [1, 12, 1000]
        r_peaks = detect_r_peaks(sig)
        record = str(chap_df.record.iloc[rec_i])
        true_labels = [CLASSES[k] for k in np.where(Y_chap[rec_i] == 1)[0]]

        fig, axes = plt.subplots(CFG.n_leads, 3, figsize=(19, 13), sharex=True)
        fig.subplots_adjust(left=0.04, right=0.93, top=0.9, bottom=0.05, hspace=0.06, wspace=0.06)
        plot_cam_column(axes[:, 0], sig, cb[0], f'Baseline {BASELINE}: final residual stage\nP({cls}) = {pb[0, c]:.2f}', r_peaks)
        plot_cam_column(axes[:, 1], sig, cf[0], f'{PROPOSED}: final IncepSE stage\nP({cls}) = {pf[0, c]:.2f}', r_peaks)
        plot_cam_column(axes[:, 2], sig, cs[0], f'{PROPOSED}: Stage-1 (lead-resolved)\nper-lead Grad-CAM', r_peaks)
        cax = fig.add_axes([0.94, 0.3, 0.008, 0.4])
        fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(0, 1), cmap='Reds'), cax=cax, label='Grad-CAM (normalized)')
        fig.suptitle(f'Chapman {record}  |  true labels: {true_labels}  |  Grad-CAM target class: {cls}  '
                     f'|  grey lines = detected R-peaks', fontsize=11)
        savefig(fig, f'fig07a_gradcam_{cls}_{record}')

In [ ]:
# =====================================================================
# Cell 7.3: Quantitative saliency metrics + Lead-SE / DKS attention capture (Chapman positives)
# =====================================================================
xai_rows, LEAD_CAM_SHARE, LEAD_SE, DKS_BRANCH, DKS_LEAD_PREF, DKS_DYNAMIC = [], {}, {}, {}, {}, {}
eval_classes = [c for c in CLASSES if y_ext[:, CLASSES.index(c)].sum() > 0]
for cls in eval_classes:
    c = CLASSES.index(cls)
    pos = np.where(y_ext[:, c] == 1)[0]
    sel = np.sort(np.random.default_rng(100 + c).choice(pos, size=min(CFG.xai_n_per_class, len(pos)), replace=False))
    Xs = np.stack([np.array(X_chap[idx_ext[p]], dtype=np.float32) for p in sel])          # [n, 12, 1000]
    cb, _ = batched_cam(cam_base, Xs, c)                                                 # [n, 1, T]
    cf, _ = batched_cam(cam_prop, Xs, c)                                                 # [n, 1, T]
    cs, _ = batched_cam(cam_s1, Xs, c, lead_groups=CFG.n_leads)                          # [n, 12, T]
    st = {key: pd.DataFrame([saliency_stats(Xs[i], cam_[i]) for i in range(len(sel))])
          for key, cam_ in [('base', cb[:, 0]), ('prop', cf[:, 0]), ('s1', cs.sum(1))]}
    row = {'class': cls, 'n': len(sel)}
    for metric in ('qrs_enrich', 'pqrst_enrich', 'off_wave_frac'):
        a, b = st['base'][metric].values, st['prop'][metric].values
        ok = np.isfinite(a) & np.isfinite(b)
        try:
            p = wilcoxon(a[ok], b[ok]).pvalue if ok.sum() >= 6 and np.any(a[ok] != b[ok]) else np.nan
        except ValueError:
            p = np.nan
        row[f'{metric}|{BASELINE}'] = np.nanmedian(a) if ok.any() else np.nan
        row[f'{metric}|{PROPOSED}'] = np.nanmedian(b) if ok.any() else np.nan
        row[f'{metric}|{PROPOSED} Stage-1'] = (np.nanmedian(st['s1'][metric].values)
                                               if np.isfinite(st['s1'][metric]).any() else np.nan)
        row[f'{metric}|Wilcoxon p (base vs prop)'] = p
    xai_rows.append(row)

    # Lead-resolved Grad-CAM share per lead (class-specific)
    tot = cs.sum((1, 2))
    good = tot > 0
    LEAD_CAM_SHARE[cls] = (cs.sum(-1)[good] / tot[good, None]).mean(0) if good.any() else np.full(CFG.n_leads, np.nan)
    # Lead-SE gates + DKS branch attention (class-agnostic gates, averaged over this class's positives)
    with ActivationCapture(prop_model.stage1.lead_gate) as cap_w, ActivationCapture(prop_model.stage1.dks_softmax) as cap_a:
        with torch.no_grad():
            for s in range(0, len(Xs), 64):
                prop_model(torch.from_numpy(Xs[s:s + 64]).to(DEVICE))
    LEAD_SE[cls] = cap_w.cat().mean(0).numpy()                                           # [12]
    att = cap_a.cat()                                                                     # [n, K, C] DKS softmax weights
    DKS_BRANCH[cls] = att.mean(2).mean(0).numpy()                                        # [K] channel-averaged (lossy)
    att_lead = att.view(att.shape[0], att.shape[1], CFG.n_leads, -1).mean(-1).mean(0)    # [K, 12] per-lead average
    DKS_LEAD_PREF[cls] = (att_lead[0] - att_lead[-1]).numpy()                            # [12] QRS-branch minus P/T-branch
    DKS_DYNAMIC[cls] = float((att.amax(0) - att.amin(0)).mean())                         # per-channel range over records

XAI_TABLE = pd.DataFrame(xai_rows).set_index(['class', 'n'])
XAI_TABLE.columns = pd.MultiIndex.from_tuples([tuple(col.split('|')) for col in XAI_TABLE.columns])
print('Saliency metrics on Chapman positives (median per class; enrichment > 1 = focus above chance):')
show(XAI_TABLE.round(3))
XAI_TABLE.to_csv(TAB_DIR / 'xai_saliency_metrics.csv')

In [ ]:
# =====================================================================
# Cell 7.4: Lead importance heatmaps (Lead-SE gates vs lead-resolved Grad-CAM share)
# =====================================================================
LEAD_SE_DF = pd.DataFrame(LEAD_SE, index=LEADS).T
LEAD_CAM_DF = pd.DataFrame(LEAD_CAM_SHARE, index=LEADS).T
fig, axes = plt.subplots(1, 2, figsize=(16, 0.45 * len(eval_classes) + 2.2))
for ax, D, title in [(axes[0], LEAD_SE_DF, 'Lead-SE gate (mean sigmoid weight): class-agnostic'),
                     (axes[1], LEAD_CAM_DF, 'Lead-resolved Grad-CAM share: class-specific')]:
    im = ax.imshow(D.values, aspect='auto', cmap='Blues')
    ax.set_xticks(range(CFG.n_leads)); ax.set_xticklabels(LEADS)
    ax.set_yticks(range(len(D))); ax.set_yticklabels(D.index)
    ax.grid(False); ax.set_title(title)
    for i in range(D.shape[0]):                                   # top-3 leads per class ko mark karo
        for j in np.argsort(-np.nan_to_num(D.values[i], nan=-1))[:3]:
            ax.scatter(j, i, s=14, color='white', edgecolor=INK, linewidth=0.6)
    fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
fig.suptitle(f'{PROPOSED}: which leads matter? (Chapman positives per class; ● = top-3 leads)', y=1.02)
savefig(fig, 'fig07b_lead_importance')
LEAD_SE_DF.to_csv(TAB_DIR / 'xai_lead_se_gates.csv')
LEAD_CAM_DF.to_csv(TAB_DIR / 'xai_lead_gradcam_share.csv')

TEXTBOOK = {'RBBB': 'V1, V2, I, V6', 'LBBB': 'I, aVL, V5, V6, V1', 'AFIB': 'II, V1 (+all)', 'AFL': 'II, III, aVF, V1',
            'PAC': 'II, V1', 'PVC': 'II, V1', 'STach': 'II', 'SBrady': 'II', 'NSR': 'II',
            'MI': 'territory-specific Q/ST', 'CD': 'V1, V6, I', 'HYP': 'V1, V5, V6', 'STTC': 'V4-V6, I, aVL', 'NORM': 'all'}
top3 = pd.DataFrame({'Grad-CAM top-3 (class-specific)': LEAD_CAM_DF.apply(lambda r: ', '.join(r.nlargest(3).index), axis=1),
                     'Lead-SE top-3 (class-agnostic)': LEAD_SE_DF.apply(lambda r: ', '.join(r.nlargest(3).index), axis=1),
                     'Textbook expectation': [TEXTBOOK.get(c, '-') for c in LEAD_CAM_DF.index]})
show(top3)

In [ ]:
# =====================================================================
# Cell 7.5: DKS branch attention per class + Stage-1 filter spectra (init vs learned)
# =====================================================================
K_SIZES = prop_model.stage1.kernel_sizes
DKS_DF = pd.DataFrame(DKS_BRANCH, index=[f'k={k} ({10 * k} ms)' for k in K_SIZES]).T
DKS_DF['dynamic range'] = pd.Series(DKS_DYNAMIC)
print('DKS branch attention per class (channel-averaged; rows sum to 1, 1/3 = no preference)')
print('+ dynamic range = mean per-channel (max - min) branch weight across records (≈0 => static DKS):')
show(DKS_DF.round(3))
DKS_DF.to_csv(TAB_DIR / 'xai_dks_branch_attention.csv')
DKS_PREF_DF = pd.DataFrame(DKS_LEAD_PREF, index=LEADS).T
DKS_PREF_DF.to_csv(TAB_DIR / 'xai_dks_lead_preference.csv')


def branch_response(conv_weight: np.ndarray, n_fft: int = 512) -> Tuple[np.ndarray, np.ndarray, float]:
    """conv_weight [C, 1, k] -> (freqs, mean normalized |H(f)|, spectral centroid Hz)."""
    H = np.abs(np.fft.rfft(conv_weight[:, 0, :], n=n_fft, axis=-1))
    H = H / np.clip(H.max(axis=1, keepdims=True), 1e-12, None)
    f = np.fft.rfftfreq(n_fft, d=1 / CFG.fs)
    Hm = H.mean(0)
    return f, Hm, float((f * Hm ** 2).sum() / (Hm ** 2).sum())


init_s1 = PhysioDKSFilter(CFG.n_leads, CFG.dks_filters_per_lead, physio_init=True)
fig, axes = plt.subplots(2, 2, figsize=(16, 0.42 * len(DKS_DF) + 7.5),
                         gridspec_kw={'height_ratios': [1, max(1.0, 0.14 * len(DKS_DF))]})
spec_rows = []
for b, k in enumerate(K_SIZES):
    for ax, conv, tag in [(axes[0, 0], init_s1.branches[b], 'init'), (axes[0, 1], prop_model.stage1.branches[b], 'learned')]:
        f, Hm, cen = branch_response(conv.weight.detach().cpu().numpy())
        ax.plot(f, Hm, color=BRANCH_COLORS[b], label=f'k={k} ({10 * k} ms)')
        spec_rows.append({'branch': f'k={k}', 'state': tag, 'spectral centroid (Hz)': round(cen, 2)})
for ax, title in [(axes[0, 0], 'Stage-1 filters at init (Gaussian-derivative wavelets)'),
                  (axes[0, 1], 'Stage-1 filters after training')]:
    ax.set_xlim(0, CFG.fs / 2); ax.set_ylim(0, 1.05)
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Mean normalized |H(f)|'); ax.set_title(title, fontsize=10)
axes[0, 1].legend(loc='upper right')

D = DKS_DF[[c for c in DKS_DF.columns if c.startswith('k=')]]
im = axes[1, 0].imshow(D.values, aspect='auto', cmap='Blues', vmin=0, vmax=max(0.5, float(np.nanmax(D.values))))
axes[1, 0].set_xticks(range(D.shape[1])); axes[1, 0].set_xticklabels(D.columns)
axes[1, 0].set_yticks(range(len(D))); axes[1, 0].set_yticklabels(D.index)
axes[1, 0].grid(False); axes[1, 0].set_title('DKS branch attention per class (channel-averaged)', fontsize=10)
for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        axes[1, 0].text(j, i, f'{D.values[i, j]:.2f}', ha='center', va='center', fontsize=8,
                        color='white' if D.values[i, j] > 0.45 else INK)
fig.colorbar(im, ax=axes[1, 0], fraction=0.05, pad=0.02)

r = float(np.nanmax(np.abs(DKS_PREF_DF.values))) if np.isfinite(DKS_PREF_DF.values).any() else 0.1
im2 = axes[1, 1].imshow(DKS_PREF_DF.values, aspect='auto', cmap='RdBu_r', vmin=-r, vmax=r)
axes[1, 1].set_xticks(range(CFG.n_leads)); axes[1, 1].set_xticklabels(LEADS)
axes[1, 1].set_yticks(range(len(DKS_PREF_DF))); axes[1, 1].set_yticklabels(DKS_PREF_DF.index)
axes[1, 1].grid(False)
axes[1, 1].set_title(f'Per-lead branch preference: k={K_SIZES[0]} − k={K_SIZES[-1]} weight\n(red = QRS-scale, blue = P/T-scale)',
                     fontsize=10)
fig.colorbar(im2, ax=axes[1, 1], fraction=0.05, pad=0.02)
fig.tight_layout()
savefig(fig, 'fig07c_dks_spectra_attention')
SPEC = pd.DataFrame(spec_rows).pivot(index='branch', columns='state', values='spectral centroid (Hz)')
print('Spectral centroid per branch (did the branches stay in their physiological bands?):')
show(SPEC)

for _c in (cam_base, cam_prop, cam_s1):
    _c.remove()                                                         # hooks clean-up
print(f'\n✅ XAI done. Figures -> {FIG_DIR.resolve()} | Tables -> {TAB_DIR.resolve()}')

---
# 8. Results ko kaise padhein aur paper mein kaise likhein 📝

## 8.1 Reading checklist (har claim ke liye evidence)

| Claim | Kahan dekho | Claim tabhi valid hai jab... |
|---|---|---|
| Proposed in-domain better hai | Cell 6.4 benchmark + significance table | `Δ Macro-AUC > 0` **aur** 95% CI 0 ko cross na kare (p < 0.05) |
| Proposed better generalize karta hai | Significance table, row "ΔAUC gap" | Gap difference negative ho aur CI 0 cross na kare |
| SE ka contribution | InceptionTime vs IncepSE | Same test set, paired bootstrap |
| Stage-1 ka contribution | IncepSE vs PhysioDKS-IncepSE (+ extended ablation) | Har component hatane pe consistent drop |
| Threshold transfer problem hai | Benchmark column "Chapman F1 (oracle thr)" vs Macro-F1, aur fig06d | Oracle − deployable gap bada ho |
| Model waves pe focus karta hai | XAI table (Cell 7.3) | Enrichment > 1, aur Wilcoxon p < 0.05 (class-conditional!) |
| Lead attention clinically sensible hai | Cell 7.4 top-3 table | Top leads textbook expectation se match karein (jaise RBBB → V1/V2) |

## 8.2 Paper ke liye honest reporting rules

1. **Single seed se strong claim mat karo.** Final paper ke liye `CFG.seeds = (42, 43, 44)` chalao aur mean ± std report karo.
2. **Primary metric pehle declare karo** (Macro-AUC), aur baad mein metrics cherry-pick mat karo.
3. **Chapman = PhysioNet `ecg-arrhythmia` v1.0.0** (Chapman-Shaoxing + Ningbo). Label caveats (Cell 2.3) "Limitations" mein likho.
4. **XAI = evidence, proof nahi.** Grad-CAM ek approximation hai. "Visually proves" ki jagah "quantitative saliency analysis indicates..." likho.
5. **Negative result bhi result hai.** Agar ΔAUC kam nahi hua, to report karo aur explain karo (jaise label shift, threshold shift).

## 8.3 Next steps (research roadmap)

| Priority | Experiment | Kyun |
|---|---|---|
| 1 | 3 seeds + `RUN_EXTENDED_ABLATION = True` | Statistically solid ablation table |
| 2 | `LABEL_SET = 'superclass5'` | PTB-XL literature se comparable in-domain numbers |
| 3 | Loss ablation: `CFG.loss_name = 'focal'` / `'asl'` (alag `run_dir`) | Imbalance handling ka effect |
| 4 | Reverse direction (Chapman → PTB-XL) | Generalization symmetric hai ya nahi |
| 5 | ECG augmentations (amplitude scaling, baseline wander, lead dropout), saare models pe same | Domain generalization ka standard baseline |
| 6 | Calibration analysis (ECE, reliability diagrams) | Threshold shift ko explain karne ke liye |

## 8.4 Reproducibility checklist ✅

- [x] Fixed seeds + deterministic cuDNN
- [x] Patient-level leakage `assert`
- [x] Chapman kabhi train / early-stopping / thresholds / normalization stats mein use nahi hua
- [x] Same preprocessing, head, optimizer, scheduler, loss, epochs, early stopping saare models ke liye
- [x] Saare tables (`tables/*.csv`), figures (`figures/*.png`, 300 dpi), thresholds (`thresholds_fold9.json`) aur predictions (`preds.npz`) disk pe saved
- [x] Library versions Cell 1 mein print hote hain (paper ke implementation details ke liye)

---

## 📚 Consolidated Glossary Index (kaunsa term kis cell mein explain hua)

| Cell | Terms |
|---|---|
| **0** | ECG/12-lead, multi-label, domain shift, source/target domain, zero-shot, class imbalance, benchmark, ablation, XAI, hypothesis |
| **1** | hyper-parameter, random seed, deterministic mode, CUDA/GPU, AMP, dataclass, cache, smoke test |
| **2** | WFDB, SCP-ECG, SNOMED-CT, label harmonization, multi-hot, superclass, `strat_fold`, patient-stratified split, data leakage, prevalence, label shift, Tier-A/B; NSR, AFIB, AFL, RBBB, LBBB, PAC, PVC, STach, SBrady |
| **3** | sampling rate, Nyquist, aliasing, resampling, FFT, Gibbs ringing, baseline wander, bandpass, Butterworth, zero-phase/filtfilt, SOS, z-score, flat lead, PSD, memmap, Dataset/DataLoader, tensor shape |
| **4** | Conv1d, kernel size, receptive field, stride/padding, channels, BatchNorm, InstanceNorm, ReLU, residual connection, ResNet, Inception module, bottleneck, max-pool, SE block, sigmoid, softmax, depthwise conv, DKS/selective kernel, RMS energy, log-energy, LayerNorm, Gaussian-derivative/Ricker wavelet, GAP, dropout, logits, parameters |
| **5** | loss, BCEWithLogits, focal loss, ASL, AdamW, weight decay, learning rate, OneCycleLR/warm-up, gradient clipping, GradScaler, epoch, validation set, early stopping/patience, checkpoint, overfitting |
| **6** | ROC, AUC, macro average, TP/FP/FN/TN, precision, F1, AUPRC, threshold tuning, bootstrap CI, paired bootstrap, p-value, generalization gap, calibration, oracle threshold |
| **7** | Grad-CAM, saliency map, activation, gradient, forward hook, lead-resolved CAM, P/QRS/T waves, R-peak, RR interval, TP segment, enrichment ratio, Wilcoxon test, frequency response, spectral centroid |

### Key references
- Wagner et al. (2020). *PTB-XL, a large publicly available electrocardiography dataset.* Scientific Data.
- Zheng et al. (2020/2022). *A 12-lead ECG database for arrhythmia research* / PhysioNet `ecg-arrhythmia` v1.0.0.
- Strodthoff et al. (2021). *Deep Learning for ECG Analysis: Benchmarks and Insights from PTB-XL.* IEEE JBHI.
- Reyna et al. (2021/2022). *Will Two Do? Varying Dimensions in ECG: The PhysioNet/CinC Challenge 2021.*
- Ismail Fawaz et al. (2020). *InceptionTime: Finding AlexNet for Time Series Classification.* DMKD.
- Hu et al. (2018). *Squeeze-and-Excitation Networks.* CVPR. · Li et al. (2019). *Selective Kernel Networks.* CVPR.
- Lee et al. (2019). *SRM: A Style-based Recalibration Module.* ICCV (mean+std squeeze).
- Ulyanov et al. (2016). *Instance Normalization.* · Pan et al. (2018). *IBN-Net* (IN for domain generalization). ECCV.
- Selvaraju et al. (2017). *Grad-CAM.* ICCV. · Lin et al. (2017). *Focal Loss.* ICCV. · Ridnik et al. (2021). *Asymmetric Loss.* ICCV.